# Objetivo

* Realizar una separación entre los individuos de la muestra entre 2 etapas etarias: Niños y adolescentes
* Realizar un LCA con k = 2-6
* Caracterizar al perfil típico (infante típico y adolescente típico) dentro de la muestra (¿Qué síntomas presenta en mayor o menor medida?)
* Analizar y reportar los resultados para cada valor de k
* Verbalizar los resultados para k = 3 y 6

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings

from utils import *
from utils import guardar_df

In [3]:
df_sintomas_disc = pd.read_excel(r"data\Sintomas_disc_Infante_Juvenil_n=1558.xlsx")
df_base = pd.read_excel(r"data\base_innominada.xlsx")
df_sub_umbrales  = pd.read_excel(r"data\sub_umbrales_jovenes_infantes.xlsx")
df_etiquetas = pd.read_excel(r"data\preguntas_disc.xlsx")

In [4]:
df_sintomas_con_edad = pd.merge(
    df_sintomas_disc,
    df_base[['inum', 'ed1']],  
    left_on='id',
    right_on='inum',
    how='left'
)

df_sintomas_con_edad = df_sintomas_con_edad.drop(columns=['inum'])

df_child = df_sintomas_con_edad[df_sintomas_con_edad['ed1'] <= 11].copy()
df_teen = df_sintomas_con_edad[df_sintomas_con_edad['ed1'] > 11].copy()

print('# Sujetos y columnas DF Child')
display(df_child.shape)

print('# Sujetos y columnas DF Teen')
display(df_teen.shape)

# Sujetos y columnas DF Child


(831, 3504)

# Sujetos y columnas DF Teen


(727, 3504)

In [5]:
q_list=question_list()

imprimir_etiquetas(q_list, df_etiquetas)


LISTADO DE ETIQUETAS (137 variables)
  1. [pso001] - Has friends
  2. [pso002] - Enjoys time with relatives of same age
  3. [pso003] - Nervous around new people in last year
  4. [pso004] - Nervous in a roup of children (party/school) in last year
  5. [pso005] - Nervous when doing things infront of other people
  6. [pga001] - Often worried before taking test/handing in assignment, past year
  7. [pga002] - Often seem worried before playing a sports game or doing an activity , pasy year
  8. [pga003] - Worried a lot about small mistakes made in projects or activities, past year
  9. [pga004] - Often worried about being on time in past year
 10. [pga005] - Often worried about having an illness in past year
 11. [pga023] - Type of person who is tense and finds it hard to relax
 12. [pga024] - In past year often been worried that they made a mistake/did something wrong
 13. [pga025] - In past year been worried that they made a fool of themselves infront of others
 14. [pga026] - Been v

In [6]:
# CHILD

df=df_child.copy()
final_df_child, prob_df_child=perform_lca_clustering(df, q_list, id_column='id')

_,_,_,_,_=analizar_distribucion(final_df_child, q_list)

resultados = analisis_estadistico_clusters(
    final_df=final_df_child,          
    id_column='id',              
    cluster_prefix='lca_k',     
    umbral_diferencia=0.10,      
    top_n_variables=25,         
    umbral_asociacion=0.15      
)

ANÁLISIS DE CLASES LATENTES (LCA)

[1] Preparando datos: 831 observaciones
[2] Procesando columnas de IMC...
    Transformando peso y estatura a categorías de IMC...
    ✓ IMC categorizado: ['IMC_pea002', 'IMC_pea003']

[3] Aplicando mapeo de códigos...
    ✓ Mapeo aplicado a 136 columnas

[4] Verificando tipos de datos...

[5] Análisis de valores faltantes (NaNs):
    pga001         :   32 NaNs (  3.85%)
    psa002         :   32 NaNs (  3.85%)
    pmd035         :   29 NaNs (  3.49%)

    Total: 93 NaNs en 113,016 celdas (0.08%)

[6] Preparando datos para LCA (todas variables como categóricas)...
    ✓ Datos preparados: 831 × 136
    ✓ Columnas codificadas: 136

[7] Realizando Análisis de Clases Latentes...

    k = 2 clases latentes...
      ✓ BIC: -134708.36, AIC: -137282.19
      ✓ Certeza promedio: 1.000
      ✓ Distribución:
          Clase 0: 99 (11.9%)
          Clase 1: 732 (88.1%)

    k = 3 clases latentes...
      ✓ BIC: -179975.93, AIC: -183839.05
      ✓ Certeza promedio

In [7]:
# TEEN
q_list=question_list()
df=df_teen.copy()
final_df_teen, prob_df_teen=perform_lca_clustering(df, q_list, id_column='id')

_,_,_,_,_=analizar_distribucion(final_df_teen, q_list)

resultados = analisis_estadistico_clusters(
    final_df=final_df_teen,          
    id_column='id',              
    cluster_prefix='lca_k',     
    umbral_diferencia=0.10,      
    top_n_variables=25,         
    umbral_asociacion=0.15      
)

ANÁLISIS DE CLASES LATENTES (LCA)

[1] Preparando datos: 727 observaciones
[2] Procesando columnas de IMC...
    Transformando peso y estatura a categorías de IMC...
    ✓ IMC categorizado: ['IMC_pea002', 'IMC_pea003']

[3] Aplicando mapeo de códigos...
    ✓ Mapeo aplicado a 136 columnas

[4] Verificando tipos de datos...

[5] Análisis de valores faltantes (NaNs):
    pso001         :    1 NaNs (  0.14%)
    pso002         :    1 NaNs (  0.14%)
    pso003         :    1 NaNs (  0.14%)
    pso004         :    1 NaNs (  0.14%)
    pso005         :    1 NaNs (  0.14%)
    pga001         :   13 NaNs (  1.79%)
    pga002         :    2 NaNs (  0.28%)
    pga003         :    2 NaNs (  0.28%)
    pga004         :    2 NaNs (  0.28%)
    pga005         :    2 NaNs (  0.28%)
    pga023         :    2 NaNs (  0.28%)
    pga024         :    2 NaNs (  0.28%)
    pga025         :    2 NaNs (  0.28%)
    pga026         :    2 NaNs (  0.28%)
    pga027         :    2 NaNs (  0.28%)
    pga028       

# Análisis de Jerarquización de Clases (LCA: k=2 a k=6)

## Evolución Jerárquica de las Clases para Infantes (ed1=[4-11])

| k de Origen | Clase de Origen | Total Obs. | Destino Principal (k+1) [% → Clase(Obs.)] | Destino Secundario (k+1) [% → Clase(Obs.)] | Otros Destinos |
|:-----------:|:-----------------:|:----------:|:-------------------------------------------:|:--------------------------------------------:|:--------------:|
| **2 → 3**   | 0                 | 99         | 70.7% → **0** (70)                         | 29.3% → **1** (29)                          | -              |
|             | 1                 | 732        | 85.2% → **2** (624)                        | 12.8% → **1** (94)                          | 2.0% → 0 (14)  |
| **3 → 4**   | 0                 | 84         | 54.8% → **3** (46)                         | 39.3% → **0** (33)                          | 6.0% → 1 (5)   |
|             | 1                 | 123        | 70.7% → **0** (87)                         | 22.8% → **1** (28)                          | 6.5% → 3 (8)   |
|             | 2                 | 624        | 88.6% → **2** (553)                        | 11.4% → **0** (71)                          | -              |
| **4 → 5**   | 0                 | 191        | 74.9% → **1** (143)                        | 12.6% → **0** (24)                          | 12.5% (4, 2)   |
|             | 1                 | 33         | 48.5% → **2** (16)                         | 24.2% → **1** (8)                           | 27.3% (4, 0)   |
|             | 2                 | 553        | 82.1% → **3** (454)                        | 17.9% → **1** (99)                          | -              |
|             | 3                 | 54         | 75.9% → **0** (41)                         | 18.5% → **4** (10)                          | 5.6% → 2 (3)   |
| **5 → 6**   | 0                 | 66         | 33.3% → **1** (22)                         | 31.8% → **3** (21)                          | 34.9% (0, 4, 2)|
|             | 1                 | 250        | 71.2% → **4** (178)                        | 21.2% → **3** (53)                          | 7.6% (5, 2, 1, Otros)|
|             | 2                 | 26         | 46.2% → **2** (12)                         | 46.2% → **3** (12)                          | 7.6% (1, 4)    |
|             | 3                 | 454        | 67.8% → **5** (308)                        | 32.2% → **4** (146)                         | -              |
|             | 4                 | 35         | 42.9% → **1** (15)                         | 28.6% → **2** (10)                          | 28.5% (3, 0)   |

**Interpretación resumida:**
*   **k=2, Clase 1**: Es la clase mayoritaria y se divide de manera muy estable. Su núcleo principal (624 obs) forma la futura **Clase 2 en k=3**, que luego se convierte en la **Clase 2 en k=4** (553 obs). Esta, a su vez, se traslada mayoritariamente a la **Clase 3 en k=5** (454 obs) y finalmente se divide entre las **Clases 5 y 4 en k=6**, siendo la **Clase 5** (308 obs) su heredera principal.
*   **k=2, Clase 0**: Es una clase pequeña y se divide de forma más equitativa, dando origen a partes de la **Clase 0 y Clase 1 en k=3**. Su trayectoria es más compleja y se mezcla con otros grupos en transiciones posteriores (k=4, Clase 3; k=5, Clase 1).
*   **Estabilidad**: Se observa una alta estabilidad en el núcleo de la clase mayoritaria a lo largo de las iteraciones (k=3 Clase 2 → k=4 Clase 2 → k=5 Clase 3), con porcentajes de retención superiores al 82%. Las divisiones se vuelven más complejas y mixtas al pasar de **k=5 a k=6**, donde la antigua clase mayoritaria (k=5, Clase 3) se divide en dos grandes grupos y varias clases pequeñas interactúan entre sí.

---

## Evolución Jerárquica de las Clases para Jóvenes (ed1=[12-17])

| k de Origen | Clase de Origen | Total Obs. | Destino Principal (k+1) [% → Clase(Obs.)] | Destino Secundario (k+1) [% → Clase(Obs.)] | Otros Destinos |
|:-----------:|:-----------------:|:----------:|:-------------------------------------------:|:--------------------------------------------:|:--------------:|
| **2 → 3**   | 0                 | 66         | 47.0% → **2** (31)                         | 45.5% → **1** (30)                          | 7.6% → 0 (5)   |
|             | 1                 | 661        | 70.5% → **0** (466)                        | 29.3% → **1** (194)                         | -              |
| **3 → 4**   | 0                 | 471        | 72.0% → **2** (339)                        | 19.7% → **1** (93)                          | 8.2% (3, 0)    |
|             | 1                 | 224        | 43.8% → **3** (98)                         | 30.4% → **2** (68)                          | 25.8% → 1 (57) |
|             | 2                 | 32         | 62.5% → **0** (20)                         | 28.1% → **1** (9)                           | 9.4% → 3 (3)   |
| **4 → 5**   | 0                 | 24         | 66.7% → **4** (16)                         | 29.2% → **0** (7)                           | 4.2% → 3 (1)   |
|             | 1                 | 159        | 50.9% → **1** (81)                         | 49.1% → **0** (78)                          | -              |
|             | 2                 | 407        | 93.6% → **2** (381)                        | 5.2% → **0** (21)                           | 1.2% → 1 (5)   |
|             | 3                 | 137        | 58.4% → **1** (80)                         | 24.1% → **3** (33)                          | 17.5% → 0 (24) |
| **5 → 6**   | 0                 | 130        | 83.8% → **5** (109)                        | 11.5% → **3** (15)                          | 4.7% (0, 2, 4) |
|             | 1                 | 166        | 42.8% → **0** (71)                         | 31.3% → **3** (52)                          | 25.9% (5, 2)   |
|             | 2                 | 381        | 88.7% → **1** (338)                        | 6.8% → **3** (26)                           | 4.5% (5, 0)    |
|             | 3                 | 34         | 52.9% → **3** (18)                         | 44.1% → **2** (15)                          | 2.9% → 0 (1)   |
|             | 4                 | 16         | 87.5% → **4** (14)                         | 6.2% → **0** (1)                            | 6.2% → 2 (1)   |

**Interpretación resumida:**
*   **k=2, Clase 1**: Es la clase mayoritaria y muestra una trayectoria principal muy estable. Su núcleo principal (466 obs) forma la **Clase 0 en k=3**, que luego se convierte en la **Clase 2 en k=4** (339 obs) y se consolida como la **Clase 2 en k=5** (381 obs) con una retención superior al 93%. Finalmente, este grupo se traslada de manera mayoritaria a la **Clase 1 en k=6** (338 obs).
*   **k=2, Clase 0**: Es una clase pequeña y se divide de forma equitativa en los inicios, dando origen a partes de las **Clases 2 y 1 en k=3**. Su descendencia sigue caminos diversos y no logra formar un núcleo grande y estable en iteraciones posteriores.
*   **Estabilidad**: La clase mayoritaria demuestra una alta estabilidad a través de las iteraciones (k=3 Clase 0 → k=4 Clase 2 → k=5 Clase 2), con porcentajes de retención superiores al 72% y llegando al 93.6%. La transición de **k=5 a k=6** es más compleja, donde la clase grande principal (k=5, Clase 2) se traslada masivamente a una nueva clase (k=6, Clase 1), mientras que las clases más pequeñas exhiben divisiones y fusiones más mixtas.

# Perfil Típico

Es importante mencionar que todos los análisis posteriores se hacen en contraposición al perfil típico de la muestra, respondiendo a qué síntomas dentro de cada clase se ven sobre y sub representados en comparación. Así, todos los síntomas que no se mencionen no se desvían de lo siguiente:

## Infante típico

###  Perfil Físico
- **IMC actual (IMC_pea002):**  
  Mayor probabilidad de estar en la categoría **underweight** (53.91%) o **normal weight** (39.35%). Es decir, el perfil típico tiende a tener un peso bajo o normal.
- **IMC más bajo en el último año (IMC_pea003):**  
  Similarmente, **underweight** (62.82%) es la categoría más frecuente.

---

###  Síntomas más frecuentes (prevalencia > 80%):
El perfil típico **sí experimenta**:
- **psa011:** Ha estado separado de su figura de apego en el último año (**82.67%**).
- **pso001:** Tiene amigos (**94.1%**).

---

###  Síntomas comunes (prevalencia 61-80%):
- **pso002:** Disfruta del tiempo con compañeros/relaciones de su misma edad (**77.74%**).

---

###  Síntomas moderadamente frecuentes (prevalencia 41-60%):
El perfil típico **podría presentar algunos** de estos, aunque no todos. Los más prevalentes en este rango son:
- **psa005:** A veces no podía dormirse sin la figura de apego presente (**51.5%**).
- **psa006:** Fue invitado a pasar la noche fuera de casa (**50.42%**).
- **pad029:** A menudo interrumpía a otros (**50.3%**).
- **pod002:** Discutía/replicaba a sus cuidadores (**48.26%**).

Esto sugiere que el perfil típico tiene **dificultades leves a moderadas en áreas de ansiedad de separación, impulsividad (TDAH) y conducta oposicionista**.

---

###  Síntomas poco frecuentes (prevalencia 0-20%):
El perfil típico **NO presenta** la gran mayoría de los síntomas psiquiátricos graves o externalizantes, como:
- **Síntomas psicóticos** (psz001-psz014) → casi todos < 1%.
- **Conducta antisocial grave** (pcd008, pcd009, pcd029, etc.) → casi todos < 1%.
- **Consumo de sustancias** (alcohol, marihuana, tabaco, otras drogas) → muy bajos (< 1% en la mayoría).
- **Ideación/conducta suicida** (pmd021, pmd022) → muy bajas (1-3%).
- **Síntomas depresivos graves** (pérdida de peso, enlentecimiento psicomotor) → bajos.

---

###  Síntomas de internalización (ansiedad/depresión) más prevalentes en el perfil típico:
Aunque la mayoría de síntomas están en el rango bajo, algunos con prevalencia 30-40% podrían aparecer ocasionalmente:
- **psa001:** Quería quedarse en casa con la figura de apego (31.77%).
- **pmd001:** Períodos de tristeza/depresión (34.06%).
- **psa002:** Se sintió enfermo (37.55%).
- **pad001:** Dificultad para mantener la atención (40.79%).

---

###  Síntomas externalizantes (conducta) más prevalentes:
- **pod001:** Perdía el temperamento (39.35%).
- **pod007:** Irritable/fácilmente molestado (38.51%).
- **pad024:** Se levantaba del asiento cuando no debía (37.67%).

---

## **RESUMEN:**
- **Bajo peso o peso normal**.
- **Socialmente conectado** (tiene amigos, disfruta con compañeros).
- **Ha experimentado separación de figura de apego recientemente**.
- **Posiblemente con síntomas leves de ansiedad de separación**.
- **Algunas dificultades atencionales e impulsividad** (rasgos de TDAH).
- **Bajo nivel de conducta antisocial, consumo de sustancias y síntomas psicóticos**.
- **Puede presentar irritabilidad ocasional y episodios de tristeza**, pero no depresión mayor.
- **No presenta ideación suicida activa ni conductas autoinfligidas graves**.

---

## Adolescente típico


### ** Perfil demográfico/antropométrico:**
- **IMC actual (IMC_pea002):**  
  Mayoritariamente **normal weight** (69.33%), con minorías en underweight (14.44%) y overweight (12.10%).
- **IMC más bajo en el último año (IMC_pea003):**  
  Similarmente, **normal weight** (65.34%) es lo más frecuente, pero con un porcentaje mayor de underweight (23.80%) que en el IMC actual.

---

### ** Síntomas más frecuentes (prevalencia > 80%):**
El perfil típico **sí experimenta**:
- **pso001:** Tiene amigos (**98.07%**).
- **psa011:** Ha estado separado de su figura de apego en el último año (**91.05%**).

---

### ** Síntomas muy comunes (prevalencia 61-80%):**
- **psa006:** Fue invitado a pasar la noche fuera de casa (**76.86%**).
- **pso002:** Disfruta del tiempo con compañeros/relaciones de su misma edad (**73.28%**).
- **psa007:** Temía que algo malo le pasara a su figura de apego (**66.25%**).
- **pod009:** Se sentía injustamente tratado y se enojaba (**65.66%**).
- **pod002:** Discutía/replicaba a sus cuidadores (**64.41%**).

---

### ** Síntomas moderadamente frecuentes (prevalencia 41-60%):**
El perfil típico **presenta varios** de estos síntomas:
- **psa012:** Estuvo fuera de casa varios días (**57.85%**).
- **pga001:** Se preocupaba mucho antes de exámenes/entregas (**56.86%**).
- **pad005:** Desorganizado (**55.17%**).
- **pga004:** Preocupado por llegar a tiempo (**55.03%**).
- **pso005:** Nervioso al hacer cosas frente a otras personas (**54.41%**).
- **pmd003:** Irritable y se enojaba por pequeñas cosas (**54.34%**).
- **pmd001:** Períodos de tristeza/depresión (**52.97%**).
- **pal001:** Ha consumido alcohol al menos una vez (**51.86%**).
- **pmd018:** Dificultad para mantener la atención en tareas (**50.48%**).

**Nota:** El consumo de tabaco (pni001) también es significativo (**42.34%**).

---

### ** Síntomas menos frecuentes (prevalencia 21-40%):**
El perfil típico **podría presentar ocasionalmente**:
- **pga003:** Preocupación excesiva por pequeños errores (**39.45%**).
- **pad006:** Dificultad para terminar tareas (**39.03%**).
- **pad001:** Problemas de atención sostenida (**38.48%**).
- **pga026:** Muy preocupado por ser aceptado por otros (**37.93%**).
- **pmd016:** Baja autoestima/sentirse inferior (**37.1%**).
- **pod001:** Pérdida de temperamento (**35.59%**).
- **pmd020:** Pensamientos sobre la muerte (**34.9%**).
- **psz010:** Pensaba que la gente se reía/hablaba de él/ella (**33.38%**).
- **pea010:** Se sentía mal consigo mismo/a, gordo/a (**32.41%**).
- **pso003:** Nervioso con gente nueva (**34.30%**).

---

### ** Síntomas raros (prevalencia 0-20%):**
El sujeto típico **NO presenta** la mayoría de síntomas graves:
- **Consumo de drogas duras** (psu004, psu005, psu006, etc.) → casi todos < 1%.
- **Conducta antisocial grave** (robo con violencia, uso de armas, etc.) → muy baja (< 1-4%).
- **Síntomas psicóticos** (psz013, psz014) → muy bajos (2-3%).
- **Ideación/conducta suicida** (pmd021, pmd022) → bajas (~10-11%).
- **Consumo de marihuana** (pmj001) → 13.52%.

---

## **RESUMEN:**
- **Peso normal** (a diferencia de la muestra anterior donde predominaba bajo peso).
- **Altamente social** (casi universalmente tiene amigos y disfruta con pares).
- **En proceso de independización** (separación frecuente de figuras de apego, noches fuera de casa).
- **Síntomas internalizantes significativos**:  
  - **Ansiedad de separación** (miedo por la figura de apego).  
  - **Ansiedad social/rendimiento** (nerviosismo frente a otros, preocupación por exámenes).  
  - **Síntomas depresivos** (tristeza, irritabilidad, baja autoestima).  
- **Síntomas externalizantes moderados**:  
  - **Conducta oposicionista** (discute con cuidadores, se rebela).  
  - **Problemas atencionales/impulsividad** (TDAH leve a moderado).  
- **Consumo de sustancias**:  
  - **Alcohol y tabaco** son relativamente comunes (>40%).  
  - **Drogas ilegales** son muy raras.  
- **Bajo riesgo de psicosis o conducta antisocial grave**.

---

##  Infante típico vs Adolescente típico

| **CARACTERÍSTICA** | **INFANTE TÍPICO** | **ADOLESCENTE TÍPICO** |
|-------------------|-------------------|------------------------|
| **PERFIL FÍSICO (IMC)** | Predominio **bajo peso** (53.91% underweight) | Predominio **peso normal** (69.33% normal weight) |
| **SOCIALIZACIÓN** | • Tiene amigos (94.1%)<br>• Disfruta con pares (77.74%) | • Tiene amigos casi universalmente (98.07%)<br>• Disfruta con pares (73.28%) |
| **INDEPENDENCIA/SEPARACIÓN** | • Separación de figura de apego (82.67%)<br>• Invitado a pasar noche fuera (50.42%) | • Separación de figura de apego (91.05%)<br>• Invitado a pasar noche fuera (76.86%)<br>• Fuera de casa varios días (57.85%) |
| **SÍNTOMAS INTERNALIZANTES** | • **Ansiedad de separación leve** (31.77% quería quedarse en casa)<br>• **Síntomas depresivos leves** (34.06% períodos de tristeza)<br>• Preocupaciones somáticas menores (37.55% se sentía enfermo) | • **Ansiedad de separación significativa** (66.25% miedo por figura de apego)<br>• **Ansiedad de rendimiento** (56.86% preocupación por exámenes)<br>• **Ansiedad social** (54.41% nervioso frente a otros)<br>• **Síntomas depresivos moderados** (52.97% tristeza, 54.34% irritabilidad)<br>• **Baja autoestima** (37.1% se siente inferior) |
| **SÍNTOMAS EXTERNALIZANTES** | • **Oposicionismo leve** (48.26% discutía con cuidadores)<br>• **Irritabilidad** (39.35% pérdida de temperamento)<br>• **Problemas atencionales** (40.79% dificultad de atención)<br>• **Impulsividad** (37.67% se levantaba del asiento) | • **Oposicionismo moderado** (64.41% discutía con cuidadores, 65.66% se sentía injustamente tratado)<br>• **Problemas atencionales** (50.48% dificultad para mantener atención)<br>• **Desorganización** (55.17%)<br>• **Impulsividad** (interrumpía 31.72%) |
| **CONSUMO DE SUSTANCIAS** | **Muy bajo** (<1% para alcohol, tabaco, marihuana) | **Significativo**:<br>• Alcohol: 51.86%<br>• Tabaco: 42.34%<br>• Marihuana: 13.52% |
| **CONDUCTA DE RIESGO** | **Muy baja**:<br>• Conducta antisocial grave: <1%<br>• Ideación suicida: 1-3%<br>• Síntomas psicóticos: <1% | **Moderada**:<br>• Conducta antisocial leve-moderada: 10-27%<br>• Ideación suicida: ~10-11%<br>• Síntomas psicóticos leves: 2-21%<br>• Pensamientos sobre muerte: 34.9% |
| **RIESGO CLÍNICO** | **Bajo**: Síntomas leves, principalmente evolutivos | **Moderado**: Mayor carga sintomática, especialmente para trastornos internalizantes (ansiedad, depresión) y posible TDAH |
| **PRINCIPALES ÁREAS DE DIFICULTAD** | 1. Ansiedad de separación leve<br>2. Problemas atencionales/impulsividad<br>3. Oposicionismo incipiente | 1. Ansiedad (separación, social, rendimiento)<br>2. Síntomas depresivos<br>3. Consumo de sustancias<br>4. Oposicionismo/conflictos familiares<br>5. Problemas atencionales/organización |

---

### **Tendencias evolutivas observadas:**
1. **Aumento en consumo de sustancias** (de <1% a >40% para alcohol y tabaco)
2. **Mayor independencia física** (más separación de figuras de apego y noches fuera)
3. **Intensificación de síntomas internalizantes** (ansiedad y depresión más prevalentes y severas)
4. **Cambio en perfil físico** (de bajo peso a peso normal)
5. **Incremento en conflictividad familiar** (mayor oposicionismo y discusiones)

### **Elementos que se mantienen:**
1. **Buena socialización** con pares (alto en ambos grupos)
2. **Baja prevalencia de psicosis y conducta antisocial grave**
3. **Presencia de problemas atencionales/impulsividad** 

---

# Clasificación LCA para Infantes

## Variables más discriminatorias (k = 3)

| Variable | Cramér's V | p-value | χ² | Discriminación |
|:---------|:----------:|:-------:|:--:|:--------------:|
| pcd023 | 0.3990 | 1.87e-29 | 132 | 1.000 |
| pcd027 | 0.3778 | 1.75e-26 | 119 | 1.000 |
| psz009 | 0.3606 | 1.31e-45 | 216 | 1.000 |
| pcd003 | 0.3408 | 1.09e-21 | 97 | 1.000 |
| pcd025 | 0.3262 | 6.25e-20 | 88 | 1.000 |
| pod010 | 0.3247 | 7.72e-37 | 175 | 0.997 |
| pcd021 | 0.3166 | 5.63e-35 | 167 | 0.997 |
| pmd010 | 0.3163 | 8.93e-19 | 83 | 1.000 |
| pod008 | 0.3134 | 1.87e-18 | 82 | 1.000 |
| pod007 | 0.2950 | 1.96e-16 | 72 | 1.000 |
| pcd012 | 0.2940 | 2.52e-16 | 72 | 1.000 |
| pcd015 | 0.2936 | 2.78e-16 | 72 | 1.000 |
| pcd014 | 0.2832 | 3.39e-15 | 67 | 1.000 |
| pod011 | 0.2625 | 7.93e-24 | 115 | 0.997 |
| pcd001 | 0.2585 | 8.80e-13 | 56 | 1.000 |
| pod005 | 0.2569 | 8.36e-23 | 110 | 0.997 |
| pal001 | 0.2543 | 2.13e-12 | 54 | 1.000 |
| pmd003 | 0.2531 | 2.74e-12 | 53 | 1.000 |
| pcd002 | 0.2492 | 6.17e-12 | 52 | 1.000 |
| pmd018 | 0.2491 | 6.35e-12 | 52 | 1.000 |
| pmd002 | 0.2468 | 1.03e-11 | 51 | 1.000 |
| pmd021 | 0.2455 | 8.94e-21 | 100 | 0.997 |
| psz011 | 0.2400 | 7.89e-20 | 96 | 1.000 |
| pmd013 | 0.2375 | 6.69e-11 | 47 | 1.000 |
| pcd020 | 0.2367 | 7.81e-11 | 47 | 1.000 |

## Hallazgos principales:

### Variable más discriminante:
- **`pcd023`** – *"Haber sido físicamente cruel con un animal a propósito"*
    - **Cramér's V = 0.3990** (el más alto)
    - **p = 1.87e-29**

### Dimensiones temáticas predominantes:

####  Conducta agresiva y cruel:
- `pcd023` – Haber sido físicamente cruel con un animal a propósito
- `pcd027` – Haber estado en una pelea física en la que alguien resultó herido/podría haberse herido
- `pcd025` – Haber acosado (bullied) a alguien más pequeño que no se defendería
- `pod011` – Haberse vengado de personas en el último año arruinando sus cosas/hiriéndolas
- `pcd020` – Haber roto algo/desordenado un lugar a propósito (como romper ventanas)
- `pcd021` – Haber roto/dañado las cosas de alguien más a propósito

#### Comportamiento oposicionista y disruptivo:
- `pod010` – Haber hecho cosas malas a personas en el último año a propósito
- `pod008` – Haber parecido enojado con personas o cosas en el último año
- `pod007` – Malhumorado/fácilmente irritable en el último año
- `pod005` – Haber hecho cosas solo para molestar a personas/hacerlas enojar en el último año
- `pmd003` – A menudo malhumorado e irritable, y pequeñas cosas provocaban enojo

#### Transgresiones y deshonestidad:
- `pcd003` – Haber robado de alguien cuando no estaba/nadie miraba
- `pcd014` – Haber mentido para obtener dinero o algo más
- `pcd015` – Haber mentido para no pagar dinero o para evitar hacer algo importante
- `pcd001` – Haber robado dinero u objetos
- `pcd002` – Haber hurtado en tiendas
- `pcd012` – Haberse metido en problemas por llegar 2 horas después del toque de queda

####  Sintomatología internalizante (depresiva y de fatiga):
- `pmd010` – Haber parecido hacer cosas como caminar/hablar mucho más lento en el último año
- `pmd002` – Nada divertido/desinterés en todo/triste en el último año
- `pmd021` – Haber hablado en serio sobre suicidarse en el último año
- `pmd013` – Hacer incluso pequeñas cosas fue agotador en el último año
- `pmd018` – Problemas para mantener la mente en la escuela/otras cosas en el último año

#### Síntomas psicóticos (paranoia/ideas de referencia):**
- `psz009` – Creer que la gente te espiaba
- `psz011` – Creer que alguien conspiraba contra ti/intentaba hacerte daño

#### Consumo de sustancias (inicio temprano):**
- `pal001` – Haber tomado al menos una bebida alcohólica (excluyendo sorbos de otros)

## Categorías Sub y Sobre-Representadas

---

### **Categorías Sub y Sobre-Representadas**

### **Cluster 0 (k = 3)**
**84 observaciones (10.1%)**

**Variables distintivas:**

1. pod007:
        SOBRE 1: 'SI' - 73% (61/84) vs 39% (320/831) global, +34pp
        SUB 1: 'NO' - 27% (23/84) vs 61% (511/831) global, -34pp

     2. pmd003:
        SOBRE 1: 'SI' - 74% (62/84) vs 44% (362/831) global, +30pp
        SUB 1: 'NO' - 26% (22/84) vs 56% (469/831) global, -30pp

     3. pod008:
        SOBRE 1: 'SI' - 67% (56/84) vs 38% (315/831) global, +29pp
        SUB 1: 'NO' - 33% (28/84) vs 62% (516/831) global, -29pp

     4. pmd002:
        SOBRE 1: 'SI' - 43% (36/84) vs 17% (140/831) global, +26pp
        SUB 1: 'NO' - 57% (48/84) vs 83% (691/831) global, -26pp

     5. pmd013:
        SOBRE 1: 'SI' - 42% (35/84) vs 16% (133/831) global, +26pp
        SUB 1: 'NO' - 58% (49/84) vs 84% (698/831) global, -26pp

     6. psz009:
        SOBRE 1: 'SI' - 27% (23/84) vs 3% (23/831) global, +25pp
        SUB 1: 'NO' - 73% (61/84) vs 97% (807/831) global, -24pp

     7. pmd018:
        SOBRE 1: 'SI' - 65% (55/84) vs 41% (342/831) global, +24pp
        SUB 1: 'NO' - 35% (29/84) vs 59% (489/831) global, -24pp

     8. pmd010:
        SOBRE 1: 'SI' - 31% (26/84) vs 7% (60/831) global, +24pp
        SUB 1: 'NO' - 69% (58/84) vs 93% (771/831) global, -24pp

     9. pod005:
        SOBRE 1: 'SI' - 43% (36/84) vs 20% (169/831) global, +23pp
        SUB 1: 'NO' - 57% (48/84) vs 80% (661/831) global, -22pp

    10. pcd025:
        SOBRE 1: 'SI' - 25% (21/84) vs 7% (55/831) global, +18pp
        SUB 1: 'NO' - 75% (63/84) vs 93% (776/831) global, -18pp

    11. pcd014:
        SOBRE 1: 'SI' - 19% (16/84) vs 7% (55/831) global, +12pp
        SUB 1: 'NO' - 81% (68/84) vs 93% (776/831) global, -12pp

    12. pmd021:
        SOBRE 1: 'SI' - 15% (13/84) vs 4% (30/831) global, +12pp
        SUB 1: 'NO' - 85% (71/84) vs 96% (800/831) global, -12pp

    13. psz011:
        SOBRE 1: 'SI' - 12% (10/84) vs 1% (10/831) global, +11pp
        SUB 1: 'NO' - 88% (74/84) vs 99% (820/831) global, -11pp

    14. pcd001:
        SOBRE 1: 'SI' - 19% (16/84) vs 9% (75/831) global, +10pp
        SUB 1: 'NO' - 81% (68/84) vs 91% (756/831) global, -10pp

**Perfil general:**
El Cluster 0 representa un grupo con un perfil clínico complejo caracterizado por una elevada **irritabilidad, anhedonia y fatiga**, junto con una presencia significativa de **síntomas psicóticos (paranoia) y conductas de oposición/agresión**. Es un grupo de tamaño intermedio (10.1%) que muestra una comorbilidad notable entre sintomatología depresiva atípica (irritabilidad, fatiga) y rasgos psicóticos.

**Hallazgos principales:**
*   **Alta irritabilidad y ánimo negativo:**
    *   **pod007:** 73% está gruñón/fácilmente molesto (vs 39% global) → **+34pp**
    *   **pmd003:** 74% está a menudo gruñón e irritable (vs 44% global) → **+30pp**
    *   **pod008:** 67% parece enojado con personas/cosas (vs 38% global) → **+29pp**
*   **Anhedonia y fatiga marcadas:**
    *   **pmd002:** 43% no encuentra diversión en nada/desinteresado (vs 17% global) → **+26pp**
    *   **pmd013:** 42% se cansaba haciendo incluso cosas pequeñas (vs 16% global) → **+26pp**
*   **Síntomas psicóticos de tipo paranoide:**
    *   **psz009:** 27% cree que la gente lo espiaba (vs 3% global) → **+25pp**
    *   **psz011:** 12% cree que alguien conspiraba contra él/ella (vs 1% global) → **+11pp**
*   **Problemas de atención y cognición:**
    *   **pmd018:** 65% tiene problemas para mantener la mente en la escuela/otras cosas (vs 41% global) → **+24pp**
*   **Conductas de oposición, agresión y engaño:**
    *   **pod005:** 43% hace cosas para fastidiar a la gente (vs 20% global) → **+23pp**
    *   **pmd010:** 31% hacía cosas como caminar/hablar mucho más lento (vs 7% global) → **+24pp** (posible agitación o lentitud psicomotora)
    *   **pcd025:** 25% ha intimidado/buleado a alguien (vs 7% global) → **+18pp**
    *   **pcd014:** 19% ha mentido para obtener dinero/algo (vs 7% global) → **+12pp**

**Variables más distintivas (diferencias >20pp):**
| Variable | Descripción (Traducida) | % Cluster 0 (SÍ) | % Global (SÍ) | Diferencia |
| :--- | :--- | :--- | :--- | :--- |
| **pod007** | Gruñón/fácilmente molesto (último año) | 73% | 39% | **+34pp** |
| **pmd003** | A menudo gruñón e irritable (último año) | 74% | 44% | **+30pp** |
| **pod008** | Parece enojado con personas/cosas (último año) | 67% | 38% | **+29pp** |
| **pmd002** | Nada es divertido/desinteresado/triste (último año) | 43% | 17% | **+26pp** |
| **pmd013** | Hacer incluso cosas pequeñas era agotador (último año) | 42% | 16% | **+26pp** |
| **psz009** | Creer que la gente lo espiaba | 27% | 3% | **+25pp** |
| **pmd010** | Caminar/hablar mucho más lento (último año) | 31% | 7% | **+24pp** |
| **pod005** | Hace cosas para fastidiar/enojar a la gente (último año) | 43% | 20% | **+23pp** |

**Síntesis interpretativa:**
El Cluster 0 representa un **perfil de depresión con características mixtas**, donde predominan:
1.  **Irritabilidad como síntoma cardinal:** Es el rasgo más sobresaliente, muy por encima de la población general.
2.  **Síntomas negativos y de energía:** Marcada anhedonia (falta de interés) y fatiga severa.
3.  **Componente psicótico paranoide:** Creencias de persecución o espionaje en una proporción considerable.
4.  **Desregulación conductual asociada:** Conductas oposicionistas, agresión (bullying) y engaño.
Este patrón sugiere un cuadro que podría no ajustarse a una depresión típica, mostrando rasgos de **desregulación emocional severa con posible riesgo psicótico incipiente**. La combinación de irritabilidad, paranoia y agresión merece una evaluación clínica cuidadosa.

---

### **Cluster 1 (k = 3)**
**123 observaciones (14.8%)**

**Variables distintivas:**

1. pod005:
        SOBRE 1: 'SI' - 46% (57/123) vs 20% (169/831) global, +26pp
        SUB 1: 'NO' - 53% (65/123) vs 80% (661/831) global, -27pp

     2. pod008:
        SOBRE 1: 'SI' - 63% (77/123) vs 38% (315/831) global, +25pp
        SUB 1: 'NO' - 37% (46/123) vs 62% (516/831) global, -25pp

     3. pod011:
        SOBRE 1: 'SI' - 33% (41/123) vs 10% (82/831) global, +23pp
        SUB 1: 'NO' - 66% (81/123) vs 90% (748/831) global, -24pp

     4. pod010:
        SOBRE 1: 'SI' - 28% (35/123) vs 6% (48/831) global, +23pp
        SUB 1: 'NO' - 71% (87/123) vs 94% (782/831) global, -23pp

     5. pcd021:
        SOBRE 1: 'SI' - 27% (33/123) vs 5% (42/831) global, +22pp
        SUB 1: 'NO' - 72% (89/123) vs 95% (788/831) global, -22pp

     6. pmd018:
        SOBRE 1: 'SI' - 60% (74/123) vs 41% (342/831) global, +19pp
        SUB 1: 'NO' - 40% (49/123) vs 59% (489/831) global, -19pp

     7. pcd023:
        SOBRE 1: 'SI' - 22% (27/123) vs 4% (33/831) global, +18pp
        SUB 1: 'NO' - 78% (96/123) vs 96% (798/831) global, -18pp

     8. pod007:
        SOBRE 1: 'SI' - 55% (68/123) vs 39% (320/831) global, +17pp
        SUB 1: 'NO' - 45% (55/123) vs 61% (511/831) global, -17pp

     9. pcd027:
        SOBRE 1: 'SI' - 20% (25/123) vs 4% (34/831) global, +16pp
        SUB 1: 'NO' - 80% (98/123) vs 96% (797/831) global, -16pp

    10. pcd001:
        SOBRE 1: 'SI' - 24% (29/123) vs 9% (75/831) global, +15pp
        SUB 1: 'NO' - 76% (94/123) vs 91% (756/831) global, -15pp

    11. pmd003:
        SOBRE 1: 'SI' - 58% (71/123) vs 44% (362/831) global, +14pp
        SUB 1: 'NO' - 42% (52/123) vs 56% (469/831) global, -14pp

    12. pcd003:
        SOBRE 1: 'SI' - 16% (20/123) vs 4% (30/831) global, +13pp
        SUB 1: 'NO' - 84% (103/123) vs 96% (801/831) global, -13pp

    13. pcd014:
        SOBRE 1: 'SI' - 19% (23/123) vs 7% (55/831) global, +12pp
        SUB 1: 'NO' - 81% (100/123) vs 93% (776/831) global, -12pp

    14. pmd021:
        SOBRE 1: 'SI' - 14% (17/123) vs 4% (30/831) global, +10pp
        SUB 1: 'NO' - 85% (105/123) vs 96% (800/831) global, -11pp

    15. pcd025:
        SOBRE 1: 'SI' - 17% (21/123) vs 7% (55/831) global, +10pp
        SUB 1: 'NO' - 83% (102/123) vs 93% (776/831) global, -10pp
  


**Perfil general:**
El Cluster 1 representa un perfil de **externalización grave y agresión**, caracterizado por altas tasas de **conductas antisociales (agresión física, crueldad, vandalismo, engaño), oposición desafiante y problemas de atención**. Es el segundo grupo más grande (14.8%) y se distingue por un patrón de comportamiento disruptivo, agresivo y transgresor de normas.

**Hallazgos principales:**
*   **Conducta oposicionista y vengativa:**
    *   **pod005:** 46% hace cosas para fastidiar a la gente (vs 20% global) → **+26pp**
    *   **pod011:** 33% se venga de la gente (vs 10% global) → **+23pp**
    *   **pod010:** 28% hace cosas malas a la gente a propósito (vs 6% global) → **+23pp**
*   **Conducta antisocial agresiva y destructiva:**
    *   **pcd021:** 27% ha roto/dañado las cosas de alguien a propósito (vs 5% global) → **+22pp**
    *   **pcd023:** 22% ha sido físicamente cruel con un animal a propósito (vs 4% global) → **+18pp**
    *   **pcd027:** 20% ha estado en una pelea física con heridos (vs 4% global) → **+16pp**
*   **Engaño y robo:**
    *   **pcd014:** 19% ha mentido para obtener dinero/algo (vs 7% global) → **+12pp**
    *   **pcd003:** 16% ha robado algo de alguien sin que estuviera (vs 4% global) → **+13pp**
*   **Problemas de atención e irritabilidad:**
    *   **pmd018:** 60% tiene problemas para mantener la mente en la escuela/otras cosas (vs 41% global) → **+19pp**
    *   **pod007:** 55% está gruñón/fácilmente molesto (vs 39% global) → **+17pp**
    *   **pmd003:** 58% está a menudo gruñón e irritable (vs 44% global) → **+14pp**
*   **Sintomatología internalizante comórbida (menor intensidad):**
    *   **pcd001:** 24% se ha sentido mal consigo mismo/sobrepeso (vs 9% global) → **+15pp**
    *   **pmd021:** 14% ha hablado seriamente de suicidarse (vs 4% global) → **+10pp**

**Variables más distintivas (diferencias >20pp):**
| Variable | Descripción (Traducida) | % Cluster 1 (SÍ) | % Global (SÍ) | Diferencia |
| :--- | :--- | :--- | :--- | :--- |
| **pod005** | Hace cosas para fastidiar/enojar a la gente (último año) | 46% | 20% | **+26pp** |
| **pod008** | Parece enojado con personas/cosas (último año) | 63% | 38% | **+25pp** |
| **pod011** | Se venga de la gente (último año) | 33% | 10% | **+23pp** |
| **pod010** | Hace cosas malas a la gente a propósito (último año) | 28% | 6% | **+23pp** |
| **pcd021** | Ha roto/dañado las cosas de alguien a propósito | 27% | 5% | **+22pp** |
| **pcd023** | Ha sido físicamente cruel con un animal a propósito | 22% | 4% | **+18pp** |
| **pcd027** | Ha estado en una pelea física con heridos | 20% | 4% | **+16pp** |

**Síntesis interpretativa:**
El Cluster 1 representa un **perfil de trastorno de conducta severo** que se caracteriza por:
1.  **Agresión interpersonal y hacia la propiedad:** Altos niveles de crueldad, vandalismo y peleas físicas.
2.  **Oposición y venganza:** Patrón desafiante, molestar a otros y conductas vindicativas.
3.  **Engaño y falta de remordimientos:** Mentiras para beneficio y robo.
4.  **Problemas de atención asociados:** Dificultades significativas de concentración que pueden indicar TDAH comórbido.
Este perfil sugiere un **riesgo elevado de trastorno de conducta** con posibles implicaciones legales y sociales. La presencia de síntomas internalizantes (como ideación suicida) en niveles bajos pero elevados respecto al global indica un malestar emocional subyacente que puede alimentar la conducta agresiva.

---

### **Cluster 2 (k = 3)**
**624 observaciones (75.1%)**

No hay desviación significativa del perfil típico

## Síntesis de Diferenciación de Clases (k=3)

| **Criterio** | **Cluster 0 (Depresión Mixta con Rasgos Psicóticos)** | **Cluster 1 (Trastorno de Conducta Severo)** |
| :--- | :--- | :--- |
| **Tamaño del Cluster** | 84 observaciones (10.1%) | 123 observaciones (14.8%) |
| **Patrón Principal** | **Comorbilidad internalizante-externalizante con rasgos psicóticos.** Elevada irritabilidad, anhedonia y fatiga (síntomas depresivos atípicos), combinada con paranoia y conductas oposicionistas/agresivas. | **Externalización grave y agresión.** Altas tasas de conductas antisociales (vandalismo, crueldad, peleas), oposición desafiante, venganza y problemas de atención. |
| **Núcleo del Perfil** | **Irritabilidad-depresión con riesgo psicótico.** La **irritabilidad extrema** es el síntoma cardinal, acompañado de pérdida de interés, fatiga severa y creencias paranoides de persecución. | **Trastorno de conducta agresivo y destructivo.** Se caracteriza por un patrón de **agresión hacia la propiedad y los animales**, junto con un estilo interpersonal vengativo y desafiante. |
| **Síntomas más Distintivos (Diferencia >20pp vs. Global)** | 1. **pod007:** Gruñón/fácilmente molesto (+34pp)<br>2. **pmd003:** A menudo gruñón e irritable (+30pp)<br>3. **pod008:** Parece enojado con personas/cosas (+29pp)<br>4. **pmd002:** Nada es divertido/desinteresado (+26pp)<br>5. **pmd013:** Fatiga con esfuerzos mínimos (+26pp)<br>6. **psz009:** Creer que la gente lo espiaba (+25pp)<br>7. **pmd010:** Lentitud/agitación psicomotora (+24pp)<br>8. **pod005:** Hace cosas para fastidiar (+23pp) | 1. **pod005:** Hace cosas para fastidiar/enojar (+26pp)<br>2. **pod008:** Parece enojado con personas/cosas (+25pp)<br>3. **pod011:** Se venga de la gente (+23pp)<br>4. **pod010:** Hace cosas malas a propósito (+23pp)<br>5. **pcd021:** Daña las cosas de otros a propósito (+22pp) |
| **Riesgo Asociado** | **Moderado-Alto.** Riesgo de transición hacia trastornos psicóticos, y deterioro social por la irritabilidad y agresión asociada. La paranoia incrementa el riesgo de aislamiento y conflicto. | **Alto.** Riesgo significativo de persistencia del trastorno de conducta con implicaciones legales, fracaso académico, rechazo social y desarrollo de trastornos de personalidad antisocial. La comorbilidad con problemas de atención y malestar internalizante agrava el pronóstico. |
| **Abordaje Sugerido** | **Evaluación clínica integral y tratamiento especializado.** Se requiere:<br>1. **Evaluación psiquiátrica** para descartar trastorno depresivo mayor con características mixtas/psicóticas.<br>2. **Psicoterapia** para manejo de la irritabilidad, regulación emocional y desafiar creencias paranoides.<br>3. **Posible farmacoterapia** (estabilizadores del ánimo, antipsicóticos atípicos).<br>4. **Intervención familiar** para manejar la conducta oposicionista. | **Intervención conductual intensiva y multimodal.** Se recomienda:<br>1. **Terapia cognitivo-conductual** centrada en el control de la ira, solución de problemas y entrenamiento en habilidades sociales.<br>2. **Evaluación y tratamiento del TDAH** comórbido si se confirma.<br>3. **Intervención familiar** (ej., entrenamiento para padres) para establecer límites claros y manejar la conducta desafiante.<br>4. **Colaboración con la escuela** para manejar la agresión y el vandalismo.<br>5. **Monitoreo del riesgo internalizante** (ideación suicida). |


---



## Variables más discriminatorias (k = 6)


| Variable                                | Cramér's V | p-value   | χ²  | Discriminación |
|-----------------------------------------|------------|-----------|-----|----------------|
| pcd002                                  | 0.4822     | 8.13e-40  | 193 | 1.000          |
| pcd003                                  | 0.4617     | 2.18e-36  | 177 | 1.000          |
| pcd023                                  | 0.4220     | 3.53e-30  | 148 | 1.000          |
| pcd027                                  | 0.4098     | 2.25e-28  | 140 | 1.000          |
| pmd003                                  | 0.4049     | 1.12e-27  | 136 | 1.000          |
| psz009                                  | 0.3995     | 3.43e-51  | 265 | 0.994          |
| pod008                                  | 0.3957     | 2.25e-26  | 130 | 1.000          |
| pmd012                                  | 0.3882     | 2.41e-25  | 125 | 1.000          |
| pcd012                                  | 0.3832     | 1.16e-24  | 122 | 1.000          |
| pmd018                                  | 0.3814     | 2.05e-24  | 121 | 1.000          |
| pod007                                  | 0.3783     | 5.38e-24  | 119 | 1.000          |
| pmd007                                  | 0.3643     | 3.53e-22  | 110 | 1.000          |
| pmd008                                  | 0.3631     | 5.07e-22  | 110 | 1.000          |
| pcd022                                  | 0.3612     | 8.88e-22  | 108 | 1.000          |
| pmd013                                  | 0.3569     | 3.12e-21  | 106 | 1.000          |
| pmd006                                  | 0.3558     | 4.18e-21  | 105 | 1.000          |
| pmd001                                  | 0.3452     | 8.58e-20  | 99  | 1.000          |
| pod010                                  | 0.3444     | 6.25e-37  | 197 | 0.994          |
| pmd021                                  | 0.3423     | 2.03e-36  | 195 | 0.994          |
| pcd015                                  | 0.3361     | 1.03e-18  | 94  | 1.000          |
| pcd025                                  | 0.3279     | 9.31e-18  | 89  | 1.000          |
| pmd002                                  | 0.3242     | 2.44e-17  | 87  | 1.000          |
| pcd001                                  | 0.3176     | 1.31e-16  | 84  | 1.000          |
| pcd014                                  | 0.3130     | 4.21e-16  | 81  | 1.000          |
| pmd015                                  | 0.3098     | 3.99e-29  | 160 | 0.994          |

## Hallazgos principales:

### Variables más discriminantes (Cramér's V > 0.40):
- **`pcd002`** – *"Ever shoplifted"* (Hurto en tiendas)
    - **Cramér's V = 0.4822** (el más alto)
    - **p = 8.13e-40**
- **`pcd003`** – *"Ever stolen from someone when they weren't there/no one was watching"* (Robar de alguien cuando no está/nadie mira)
    - **Cramér's V = 0.4617**
    - **p = 2.18e-36**
- **`pcd023`** – *"Ever been physically cruel to an animal on purpose"* (Haber sido físicamente cruel con un animal a propósito)
    - **Cramér's V = 0.4220**
    - **p = 3.53e-30**
- **`pcd027`** – *"Ever been in a physical fight in which someone was hurt/could have been hurt"* (Participar en peleas físicas con lesiones o potenciales lesiones)
    - **Cramér's V = 0.4098**
    - **p = 2.25e-28**
- **`pmd003`** – *"Often grouchy and irritable and little things created anger"* (A menudo malhumorado e irritable, y pequeñas cosas provocaban enojo)
    - **Cramér's V = 0.4049**
    - **p = 1.12e-27**

### Dimensiones temáticas predominantes:

#### **Conducta antisocial y delictiva:**
- `pcd002` – Hurto en tiendas (shoplifting)
- `pcd003` – Robar a alguien en su ausencia
- `pcd001` – Robar dinero u objetos (etiqueta truncada)
- `pcd022` – Iniciar un fuego sin permiso (piromanía)
- `pcd012` – Problemas por incumplir toque de queda

#### **Agresividad y crueldad:**
- `pcd023` – Crueldad física hacia animales a propósito
- `pcd027` – Participar en peleas físicas con lesiones o potenciales lesiones
- `pcd025` – Acosar (bully) a alguien más pequeño que no se defendería
- `pod010` – Hacer cosas malas a personas intencionalmente

#### **Deshonestidad y manipulación:**
- `pcd014` – Mentir para obtener dinero o algo más
- `pcd015` – Mentir para evitar pagar dinero o para evitar hacer algo importante

#### **Sintomatología Internalizante:**
- `pmd001` – Períodos en el último año en que a menudo parecía muy angustiado/deprimido
- `pmd002` – Nada divertido/desinterés en todo/triste en el último año
- `pmd021` – Hablar seriamente sobre suicidarse en el último año
- `pmd015` – Culparse a sí mismo por cosas malas que sucedieron en el último año
- `pmd012` – Menos energía en el último año
- `pmd013` – Hacer incluso pequeñas cosas fue agotador en el último año
- `pmd008` – Problemas para dormir en el último año
- `pmd007` – Mucha más hambre/comió mucho más de lo habitual en el último año
- `pmd006` – Aumento significativo de peso en el último año
- `pmd018` – Dificultad para mantener la mente en la escuela/otras cosas en el último año

#### **Irritabilidad y labilidad emocional:**
- `pmd003` – A menudo malhumorado e irritable, y pequeñas cosas provocaban enojo
- `pod007` – Malhumorado/fácilmente irritable en el último año
- `pod008` – Parecer enojado con personas o cosas en el último año


#### **Síntomas psicóticos:**
- `psz009` – Creer que la gente te espiaba



## Variables más discriminatorias (k = 6)

### Clase 0 (k =6)

22 observaciones (2.6%)
  
  ---
  Variables distintivas:

     1. pmd008:
        SOBRE 1: 'SI' - 95% (21/22) vs 24% (197/831) global, +72pp

     2. pmd007:
        SOBRE 1: 'SI' - 86% (19/22) vs 22% (185/831) global, +64pp
        SUB 1: 'NO' - 14% (3/22) vs 78% (646/831) global, -64pp

     3. pod008:
        SOBRE 1: 'SI' - 91% (20/22) vs 38% (315/831) global, +53pp
        SUB 1: 'NO' - 9% (2/22) vs 62% (516/831) global, -53pp

     4. pmd013:
        SOBRE 1: 'SI' - 68% (15/22) vs 16% (133/831) global, +52pp
        SUB 1: 'NO' - 32% (7/22) vs 84% (698/831) global, -52pp

     5. pmd003:
        SOBRE 1: 'SI' - 95% (21/22) vs 44% (362/831) global, +52pp

     6. pod007:
        SOBRE 1: 'SI' - 86% (19/22) vs 39% (320/831) global, +48pp
        SUB 1: 'NO' - 14% (3/22) vs 61% (511/831) global, -48pp

     7. pmd015:
        SOBRE 1: 'SI' - 64% (14/22) vs 17% (139/831) global, +47pp
        SUB 1: 'NO' - 36% (8/22) vs 83% (691/831) global, -47pp

     8. pmd012:
        SOBRE 1: 'SI' - 59% (13/22) vs 16% (129/831) global, +44pp
        SUB 1: 'NO' - 41% (9/22) vs 84% (702/831) global, -44pp

     9. pmd001:
        SOBRE 1: 'SI' - 77% (17/22) vs 34% (283/831) global, +43pp
        SUB 1: 'NO' - 23% (5/22) vs 66% (548/831) global, -43pp

    10. pmd018:
        SOBRE 1: 'SI' - 82% (18/22) vs 41% (342/831) global, +41pp
        SUB 1: 'NO' - 18% (4/22) vs 59% (489/831) global, -41pp

    11. psz009:
        SOBRE 1: 'SI' - 41% (9/22) vs 3% (23/831) global, +38pp
        SUB 1: 'NO' - 59% (13/22) vs 97% (807/831) global, -38pp

    12. pcd022:
        SOBRE 1: 'SI' - 45% (10/22) vs 8% (63/831) global, +38pp
        SUB 1: 'NO' - 55% (12/22) vs 92% (768/831) global, -38pp

    13. pmd002:
        SOBRE 1: 'SI' - 55% (12/22) vs 17% (140/831) global, +38pp
        SUB 1: 'NO' - 45% (10/22) vs 83% (691/831) global, -38pp

    14. pmd006:
        SOBRE 1: 'SI' - 50% (11/22) vs 13% (105/831) global, +37pp
        SUB 1: 'NO' - 50% (11/22) vs 87% (726/831) global, -37pp

    15. pmd021:
        SOBRE 1: 'SI' - 32% (7/22) vs 4% (30/831) global, +28pp
        SUB 1: 'NO' - 68% (15/22) vs 96% (800/831) global, -28pp

    16. pcd025:
        SOBRE 1: 'SI' - 32% (7/22) vs 7% (55/831) global, +25pp
        SUB 1: 'NO' - 68% (15/22) vs 93% (776/831) global, -25pp

    17. pod010:
        SOBRE 1: 'SI' - 27% (6/22) vs 6% (48/831) global, +21pp
        SUB 1: 'NO' - 73% (16/22) vs 94% (782/831) global, -21pp

    18. pcd014:
        SOBRE 1: 'SI' - 27% (6/22) vs 7% (55/831) global, +21pp
        SUB 1: 'NO' - 73% (16/22) vs 93% (776/831) global, -21pp

    19. pcd003:
        SOBRE 1: 'SI' - 23% (5/22) vs 4% (30/831) global, +19pp
        SUB 1: 'NO' - 77% (17/22) vs 96% (801/831) global, -19pp

  #### Perfil general:
La **Clase 0** muestra un **perfil de depresión severa con características mixtas**, caracterizado por una **combinación extrema de síntomas depresivos neurovegetativos, irritabilidad marcada y algunos comportamientos externalizantes**. Constituye el **grupo más pequeño (2.6%) pero clínicamente más complejo** en términos de sintomatología afectiva.

#### Hallazgos principales:

##### **Síntomas internalizantes severos:**
- **`pmd008`**: 95% tiene problemas para dormir (vs 24% global) → **+72pp**
- **`pmd007`**: 86% tiene aumento significativo de apetito (vs 22% global) → **+64pp**
- **`pmd013`**: 68% se siente extremadamente fatigado (vs 16% global) → **+52pp**
- **`pmd012`**: 59% reporta menos energía (vs 16% global) → **+44pp**
- **`pmd006`**: 50% ha ganado mucho peso (vs 13% global) → **+37pp**

##### **Irritabilidad y labilidad emocional extrema:**
- **`pod008`**: 91% parece enojado con personas o cosas (vs 38% global) → **+53pp**
- **`pmd003`**: 95% está a menudo malhumorado e irritable (vs 44% global) → **+52pp**
- **`pod007`**: 86% está malhumorado/fácilmente irritable (vs 39% global) → **+48pp**

##### **Síntomas depresivos y afectivos graves:**
- **`pmd001`**: 77% ha tenido períodos de gran malestar/depresión (vs 34% global) → **+43pp**
- **`pmd015`**: 64% se culpa a sí mismo por cosas malas (vs 17% global) → **+47pp**
- **`pmd002`**: 55% siente desinterés/tristeza (vs 17% global) → **+38pp**
- **`pmd021`**: 32% ha hablado seriamente de suicidarse (vs 4% global) → **+28pp**

##### **Dificultades atencionales y cognitivas:**
- **`pmd018`**: 82% tiene dificultad para mantener la concentración (vs 41% global) → **+41pp**

##### **Síntomas psicóticos (paranoia):**
- **`psz009`**: 41% cree que la gente lo espía (vs 3% global) → **+38pp**

##### **Comportamientos externalizantes específicos:**
- **`pcd022`**: 45% ha iniciado fuegos sin permiso (piromanía) (vs 8% global) → **+38pp**
- **`pcd025`**: 32% ha acosado (bullied) a alguien más pequeño (vs 7% global) → **+25pp**
- **`pod010`**: 27% hace cosas malas a personas intencionalmente (vs 6% global) → **+21pp**
- **`pcd014`**: 27% ha mentido para obtener dinero o algo más (vs 7% global) → **+21pp**
- **`pcd003`**: 23% ha robado a alguien en su ausencia (vs 4% global) → **+19pp**

#### Variables más distintivas (diferencias >40pp):

| Variable | Descripción | % Clase 0 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pmd008** | Problemas para dormir | 95% | 24% | **+72pp** |
| **pmd007** | Aumento de apetito | 86% | 22% | **+64pp** |
| **pod008** | Parecer enojado con personas/cosas | 91% | 38% | **+53pp** |
| **pmd013** | Fatiga extrema | 68% | 16% | **+52pp** |
| **pmd003** | Irritabilidad y enojo por pequeñas cosas | 95% | 44% | **+52pp** |
| **pod007** | Malhumorado/fácilmente irritable | 86% | 39% | **+48pp** |
| **pmd015** | Autoculpa | 64% | 17% | **+47pp** |
| **pmd012** | Menos energía | 59% | 16% | **+44pp** |
| **pmd001** | Períodos de gran malestar/depresión | 77% | 34% | **+43pp** |
| **pmd018** | Dificultad de concentración | 82% | 41% | **+41pp** |

#### Síntesis interpretativa:

La **Clase 0** representa un **perfil clínico de depresión severa** que se caracteriza por:

1. **Síntomas depresivos marcados**: Alteraciones severas en sueño, apetito, energía y peso que sugieren un episodio depresivo mayor con características melancólicas/atípicas
2. **Irritabilidad como síntoma cardinal**: Mal humor y enojo extremos que superan incluso a los síntomas depresivos en prevalencia
3. **Cogniciones depresivas intensas**: Autoculpa, desesperanza e ideación suicida significativas
4. **Comorbilidad psicótica**: Creencias paranoides en una proporción importante del grupo
5. **Comportamientos externalizantes selectivos**: Piromanía como conducta particularmente distintiva, junto con bullying y mentiras

**Nota**: La representatividad de este grupo es extremadamente baja.

### Clase 1 (k =6) 
41 observaciones (4.9%)

  ---
  
  Variables distintivas:

     1. pod007:
        SOBRE 1: 'SI' - 68% (28/41) vs 39% (320/831) global, +30pp
        SUB 1: 'NO' - 32% (13/41) vs 61% (511/831) global, -30pp

     2. psz009:
        SOBRE 1: 'SI' - 27% (11/41) vs 3% (23/831) global, +24pp
        SUB 1: 'NO' - 73% (30/41) vs 97% (807/831) global, -24pp

     3. pod008:
        SOBRE 1: 'SI' - 61% (25/41) vs 38% (315/831) global, +23pp
        SUB 1: 'NO' - 39% (16/41) vs 62% (516/831) global, -23pp

     4. pmd003:
        SOBRE 1: 'SI' - 66% (27/41) vs 44% (362/831) global, +22pp
        SUB 1: 'NO' - 34% (14/41) vs 56% (469/831) global, -22pp

     5. pmd018:
        SOBRE 1: 'SI' - 63% (26/41) vs 41% (342/831) global, +22pp
        SUB 1: 'NO' - 37% (15/41) vs 59% (489/831) global, -22pp

     6. pmd001:
        SOBRE 1: 'SI' - 56% (23/41) vs 34% (283/831) global, +22pp
        SUB 1: 'NO' - 44% (18/41) vs 66% (548/831) global, -22pp

     7. pod010:
        SOBRE 1: 'SI' - 27% (11/41) vs 6% (48/831) global, +21pp
        SUB 1: 'NO' - 73% (30/41) vs 94% (782/831) global, -21pp

     8. pcd025:
        SOBRE 1: 'SI' - 27% (11/41) vs 7% (55/831) global, +20pp
        SUB 1: 'NO' - 73% (30/41) vs 93% (776/831) global, -20pp

     9. pmd007:
        SOBRE 1: 'SI' - 41% (17/41) vs 22% (185/831) global, +19pp
        SUB 1: 'NO' - 59% (24/41) vs 78% (646/831) global, -19pp

    10. pmd013:
        SOBRE 1: 'SI' - 34% (14/41) vs 16% (133/831) global, +18pp
        SUB 1: 'NO' - 66% (27/41) vs 84% (698/831) global, -18pp

    11. pmd002:
        SOBRE 1: 'SI' - 34% (14/41) vs 17% (140/831) global, +17pp
        SUB 1: 'NO' - 66% (27/41) vs 83% (691/831) global, -17pp

    12. pcd012:
        SOBRE 1: 'SI' - 17% (7/41) vs 1% (8/831) global, +16pp
        SUB 1: 'NO' - 83% (34/41) vs 99% (823/831) global, -16pp

#### Perfil general:
La **Clase 1** muestra un **perfil de irritabilidad y depresión moderada con rasgos psicóticos y comportamientos externalizantes**, caracterizado por una **combinación de síntomas emocionales, conductuales y cognitivos de intensidad moderada**. Constituye el **4.9% de la muestra** y representa un patrón clínico mixto.

#### Hallazgos principales:

##### **Irritabilidad y labilidad emocional:**
- **`pod007`**: 68% se muestra malhumorado/fácilmente irritable (vs 39% global) → **+30pp**
- **`pod008`**: 61% parece enojado con personas o cosas (vs 38% global) → **+23pp**
- **`pmd003`**: 66% está a menudo malhumorado e irritable (vs 44% global) → **+22pp**

##### **Síntomas psicóticos (paranoia):**
- **`psz009`**: 27% cree que la gente lo espía (vs 3% global) → **+24pp**

##### **Problemas atencionales y cognitivos:**
- **`pmd018`**: 63% tiene dificultad para mantener la concentración (vs 41% global) → **+22pp**

##### **Síntomas depresivos moderados:**
- **`pmd001`**: 56% ha tenido períodos de gran malestar/depresión (vs 34% global) → **+22pp**
- **`pmd002`**: 34% siente desinterés/tristeza (vs 17% global) → **+17pp**
- **`pmd013`**: 34% se siente fatigado (vs 16% global) → **+18pp**
- **`pmd007`**: 41% tiene aumento de apetito (vs 22% global) → **+19pp**

##### **Comportamientos externalizantes:**
- **`pod010`**: 27% hace cosas malas a personas intencionalmente (vs 6% global) → **+21pp**
- **`pcd025`**: 27% ha acosado (bullied) a alguien más pequeño (vs 7% global) → **+20pp**
- **`pcd012`**: 17% ha tenido problemas por incumplir toque de queda (vs 1% global) → **+16pp**

#### Variables más distintivas (diferencias ≥20pp):

| Variable | Descripción | % Clase 1 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pod007** | Malhumorado/fácilmente irritable | 68% | 39% | **+30pp** |
| **psz009** | Creer que la gente lo espía | 27% | 3% | **+24pp** |
| **pod008** | Parecer enojado con personas/cosas | 61% | 38% | **+23pp** |
| **pmd003** | Irritabilidad y enojo por pequeñas cosas | 66% | 44% | **+22pp** |
| **pmd018** | Dificultad de concentración | 63% | 41% | **+22pp** |
| **pmd001** | Períodos de gran malestar/depresión | 56% | 34% | **+22pp** |
| **pod010** | Hacer cosas malas a personas intencionalmente | 27% | 6% | **+21pp** |
| **pcd025** | Acosar (bully) a alguien más pequeño | 27% | 7% | **+20pp** |

#### Síntesis interpretativa:

La **Clase 1** representa un **perfil clínico de irritabilidad y depresión moderada con rasgos psicóticos y comportamientos externalizantes** que se caracteriza por:

1. **Irritabilidad como síntoma principal**: Mal humor y enojo significativos, aunque menos extremos que en la Clase 0.
2. **Síntomas depresivos moderados**: Períodos de malestar, desinterés y fatiga.
3. **Problemas atencionales**: Dificultad de concentración en una proporción importante.
4. **Síntomas psicóticos paranoides**: Creencia de ser espiado, que es un marcador de posible psicosis incipiente.
5. **Comportamientos externalizantes selectivos**: Bullying y hacer daño intencional, junto con problemas de conducta como incumplir toque de queda.

### Clase 2 (k =6)  
  27 observaciones (3.2%)

  ---
  Variables distintivas:

     1. pod010:
        SOBRE 1: 'SI' - 48% (13/27) vs 6% (48/831) global, +42pp
        SUB 1: 'NO' - 48% (13/27) vs 94% (782/831) global, -46pp

     2. pod008:
        SOBRE 1: 'SI' - 78% (21/27) vs 38% (315/831) global, +40pp
        SUB 1: 'NO' - 22% (6/27) vs 62% (516/831) global, -40pp

     3. pcd003:
        SOBRE 1: 'SI' - 41% (11/27) vs 4% (30/831) global, +37pp
        SUB 1: 'NO' - 59% (16/27) vs 96% (801/831) global, -37pp

     4. pcd002:
        SOBRE 1: 'SI' - 33% (9/27) vs 2% (14/831) global, +32pp
        SUB 1: 'NO' - 67% (18/27) vs 98% (817/831) global, -32pp

     5. pcd022:
        SOBRE 1: 'SI' - 37% (10/27) vs 8% (63/831) global, +29pp
        SUB 1: 'NO' - 63% (17/27) vs 92% (768/831) global, -29pp

     6. pcd001:
        SOBRE 1: 'SI' - 37% (10/27) vs 9% (75/831) global, +28pp
        SUB 1: 'NO' - 63% (17/27) vs 91% (756/831) global, -28pp

     7. pod007:
        SOBRE 1: 'SI' - 63% (17/27) vs 39% (320/831) global, +24pp
        SUB 1: 'NO' - 37% (10/27) vs 61% (511/831) global, -24pp

     8. pmd001:
        SOBRE 1: 'NO' - 85% (23/27) vs 66% (548/831) global, +19pp
        SUB 1: 'SI' - 15% (4/27) vs 34% (283/831) global, -19pp

     9. pcd025:
        SOBRE 1: 'SI' - 22% (6/27) vs 7% (55/831) global, +16pp
        SUB 1: 'NO' - 78% (21/27) vs 93% (776/831) global, -16pp

    10. pcd014:
        SOBRE 1: 'SI' - 22% (6/27) vs 7% (55/831) global, +16pp
        SUB 1: 'NO' - 78% (21/27) vs 93% (776/831) global, -16pp

    11. pcd023:
        SOBRE 1: 'SI' - 19% (5/27) vs 4% (33/831) global, +15pp
        SUB 1: 'NO' - 81% (22/27) vs 96% (798/831) global, -15pp

    12. pcd027:
        SOBRE 1: 'SI' - 19% (5/27) vs 4% (34/831) global, +14pp
        SUB 1: 'NO' - 81% (22/27) vs 96% (797/831) global, -14pp

    13. pmd006:
        SOBRE 1: 'NO' - 100% (27/27) vs 87% (726/831) global, +13pp

    14. psz009:
        SUB 1: 'NO' - 85% (23/27) vs 97% (807/831) global, -12pp

    15. pmd012:
        SOBRE 1: 'NO' - 96% (26/27) vs 84% (702/831) global, +12pp

    16. pmd021:
        SUB 1: 'NO' - 85% (23/27) vs 96% (800/831) global, -11pp
  
#### Perfil general:
La **Clase 2** muestra un **perfil de externalización grave con conductas antisociales múltiples y baja sintomatología internalizante**, caracterizado por un **patrón de agresión intencional, delincuencia y hostilidad sin comorbilidad depresiva significativa**. Constituye el **3.2% de la muestra** y representa un patrón clínico de conducta antisocial "pura".

#### Hallazgos principales:

##### **Agresividad intencional y hostilidad:**
- **`pod010`**: 48% hace cosas malas a personas intencionalmente (vs 6% global) → **+42pp**
- **`pod008`**: 78% parece enojado con personas o cosas (vs 38% global) → **+40pp**
- **`pod007`**: 63% se muestra malhumorado/fácilmente irritable (vs 39% global) → **+24pp**

##### **Conductas delictivas y antisociales graves:**
- **`pcd003`**: 41% ha robado a alguien cuando no estaba/nadie miraba (vs 4% global) → **+37pp**
- **`pcd002`**: 33% ha hurtado en tiendas (vs 2% global) → **+32pp**
- **`pcd022`**: 37% ha iniciado fuegos sin permiso (piromanía) (vs 8% global) → **+29pp**
- **`pcd001`**: 37% ha robado dinero u objetos (vs 9% global) → **+28pp**
- **`pcd025`**: 22% ha acosado (bullied) a alguien más pequeño (vs 7% global) → **+16pp**
- **`pcd023`**: 19% ha sido físicamente cruel con animales (vs 4% global) → **+15pp**
- **`pcd027`**: 19% ha estado en peleas físicas con lesiones (vs 4% global) → **+14pp**
- **`pcd014`**: 22% ha mentido para obtener dinero o algo más (vs 7% global) → **+16pp**

##### **Baja sintomatología depresiva y psicótica:**
- **`pmd001`**: Solo 15% ha tenido períodos de gran malestar/depresión (vs 34% global) → **-19pp**
- **`pmd021`**: Solo 15% ha hablado seriamente de suicidarse (vs 4% global, pero sub-representación del 85% vs 96%) → **-11pp**
- **`psz009`**: 85% NO cree que la gente lo espía (vs 97% global) → **-12pp**
- **`pmd012`**: 96% NO reporta menos energía (vs 84% global) → **+12pp** (en 'NO')
- **`pmd006`**: 100% NO ha ganado mucho peso (vs 87% global) → **+13pp** (en 'NO')

#### Variables más distintivas (diferencias >30pp):

| Variable | Descripción | % Clase 2 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pod010** | Hacer cosas malas a personas intencionalmente | 48% | 6% | **+42pp** |
| **pod008** | Parecer enojado con personas/cosas | 78% | 38% | **+40pp** |
| **pcd003** | Robar a alguien en su ausencia | 41% | 4% | **+37pp** |
| **pcd002** | Hurto en tiendas | 33% | 2% | **+32pp** |
| **pcd022** | Iniciar fuegos sin permiso (piromanía) | 37% | 8% | **+29pp** |
| **pcd001** | Robar dinero u objetos | 37% | 9% | **+28pp** |

#### Síntesis interpretativa:

La **Clase 2** representa un **perfil de externalización grave con conductas antisociales múltiples y baja comorbilidad internalizante** que se caracteriza por:

1. **Agresividad intencional y hostilidad**: Altos niveles de hacer daño intencional, enojo y mal humor.
2. **Patrón delictivo diverso**: Hurto, robo, piromanía, crueldad animal, peleas y bullying.
3. **Deshonestidad instrumental**: Mentiras para obtener beneficios.
4. **Baja sintomatología depresiva**: Significativamente menos síntomas depresivos, ideación suicida y fatiga.
5. **Ausencia de síntomas psicóticos**: No hay creencias paranoides significativas.

**Características distintivas importantes**:
- **Agresión instrumental**: El daño intencional (`pod010`) es extremadamente alto.
- **Patrón delictivo diverso**: Incluye desde hurtos hasta piromanía.
- **Bajo afecto negativo**: Menos síntomas depresivos que la población general.
- **Hostilidad crónica**: Enojo y mal humor persistentes.

Este perfil representa un **subgrupo de externalización grave relativamente "pura"**, que podría corresponder a un trastorno de conducta más clásico sin comorbilidad emocional significativa. La falta de síntomas depresivos puede indicar menor angustia subjetiva pero mayor riesgo de conductas antisociales persistentes.

### Clase 3 (k =6)  
  94 observaciones (11.3%)
  
  ---
  Variables distintivas:

     1. pmd018:
        SOBRE 1: 'SI' - 68% (64/94) vs 41% (342/831) global, +27pp
        SUB 1: 'NO' - 32% (30/94) vs 59% (489/831) global, -27pp

     2. pmd003:
        SOBRE 1: 'SI' - 67% (63/94) vs 44% (362/831) global, +23pp
        SUB 1: 'NO' - 33% (31/94) vs 56% (469/831) global, -23pp

     3. pod008:
        SOBRE 1: 'SI' - 60% (56/94) vs 38% (315/831) global, +22pp
        SUB 1: 'NO' - 40% (38/94) vs 62% (516/831) global, -22pp

     4. pcd023:
        SOBRE 1: 'SI' - 24% (23/94) vs 4% (33/831) global, +20pp
        SUB 1: 'NO' - 76% (71/94) vs 96% (798/831) global, -20pp

     5. pcd027:
        SOBRE 1: 'SI' - 23% (22/94) vs 4% (34/831) global, +19pp
        SUB 1: 'NO' - 77% (72/94) vs 96% (797/831) global, -19pp

     6. pod007:
        SOBRE 1: 'SI' - 57% (54/94) vs 39% (320/831) global, +19pp
        SUB 1: 'NO' - 43% (40/94) vs 61% (511/831) global, -19pp

     7. pmd021:
        SOBRE 1: 'SI' - 21% (20/94) vs 4% (30/831) global, +18pp
        SUB 1: 'NO' - 79% (74/94) vs 96% (800/831) global, -18pp

     8. pmd002:
        SOBRE 1: 'SI' - 32% (30/94) vs 17% (140/831) global, +15pp
        SUB 1: 'NO' - 68% (64/94) vs 83% (691/831) global, -15pp

     9. pmd001:
        SOBRE 1: 'SI' - 49% (46/94) vs 34% (283/831) global, +15pp
        SUB 1: 'NO' - 51% (48/94) vs 66% (548/831) global, -15pp

    10. pcd001:
        SOBRE 1: 'SI' - 23% (22/94) vs 9% (75/831) global, +14pp
        SUB 1: 'NO' - 77% (72/94) vs 91% (756/831) global, -14pp

    11. pcd014:
        SOBRE 1: 'SI' - 20% (19/94) vs 7% (55/831) global, +14pp
        SUB 1: 'NO' - 80% (75/94) vs 93% (776/831) global, -14pp

    12. pcd015:
        SOBRE 1: 'SI' - 16% (15/94) vs 3% (23/831) global, +13pp
        SUB 1: 'NO' - 84% (79/94) vs 97% (808/831) global, -13pp

    13. pmd008:
        SOBRE 1: 'SI' - 34% (32/94) vs 24% (197/831) global, +10pp
        SUB 1: 'NO' - 66% (62/94) vs 76% (634/831) global, -10pp

#### Perfil general:
La **Clase 3** muestra un **perfil mixto de problemas atencionales con irritabilidad, conductas antisociales y síntomas depresivos**, caracterizado por una **combinación de dificultades cognitivas, emocionales y conductuales de intensidad moderada**. Constituye el **11.3% de la muestra** y representa un patrón clínico de múltiples dominios afectados.

#### Hallazgos principales:

##### **Problemas atencionales y cognitivos prominentes:**
- **`pmd018`**: 68% tiene dificultad para mantener la concentración (vs 41% global) → **+27pp**

##### **Irritabilidad y hostilidad significativa:**
- **`pmd003`**: 67% está a menudo malhumorado e irritable (vs 44% global) → **+23pp**
- **`pod008`**: 60% parece enojado con personas o cosas (vs 38% global) → **+22pp**
- **`pod007`**: 57% se muestra malhumorado/fácilmente irritable (vs 39% global) → **+19pp**

##### **Conductas antisociales diversas:**
- **`pcd023`**: 24% ha sido físicamente cruel con animales (vs 4% global) → **+20pp**
- **`pcd027`**: 23% ha estado en peleas físicas con lesiones (vs 4% global) → **+19pp**
- **`pcd001`**: 23% ha robado dinero u objetos (vs 9% global) → **+14pp**
- **`pcd014`**: 20% ha mentido para obtener dinero o algo más (vs 7% global) → **+14pp**
- **`pcd015`**: 16% ha mentido para evitar pagar o hacer algo importante (vs 3% global) → **+13pp**

##### **Síntomas depresivos y de ideación suicida:**
- **`pmd021`**: 21% ha hablado seriamente de suicidarse (vs 4% global) → **+18pp**
- **`pmd002`**: 32% siente desinterés/tristeza (vs 17% global) → **+15pp**
- **`pmd001`**: 49% ha tenido períodos de gran malestar/depresión (vs 34% global) → **+15pp**
- **`pmd008`**: 34% tiene problemas para dormir (vs 24% global) → **+10pp**

#### Variables más distintivas (diferencias >15pp):

| Variable | Descripción | % Clase 3 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pmd018** | Dificultad de concentración | 68% | 41% | **+27pp** |
| **pmd003** | Irritabilidad y enojo por pequeñas cosas | 67% | 44% | **+23pp** |
| **pod008** | Parecer enojado con personas/cosas | 60% | 38% | **+22pp** |
| **pcd023** | Crueldad física hacia animales | 24% | 4% | **+20pp** |
| **pcd027** | Peleas físicas con lesiones | 23% | 4% | **+19pp** |
| **pod007** | Malhumorado/fácilmente irritable | 57% | 39% | **+19pp** |
| **pmd021** | Hablar seriamente de suicidarse | 21% | 4% | **+18pp** |

#### Síntesis interpretativa:

La **Clase 3** representa un **perfil clínico complejo con problemas atencionales prominentes, irritabilidad, conductas antisociales y síntomas depresivos** que se caracteriza por:

1. **Dificultades atencionales como síntoma principal**: Problemas de concentración son el marcador más distintivo.
2. **Irritabilidad y hostilidad significativas**: Mal humor, irritabilidad y enojo persistentes.
3. **Conductas antisociales diversas**: Crueldad animal, peleas, robos y mentiras.
4. **Sintomatología depresiva con riesgo suicida**: Ideación suicida, desinterés y períodos de malestar depresivo.
5. **Alteraciones del sueño**: Problemas para dormir en un tercio de la clase.

**Características distintivas importantes**:
- **Problemas atencionales prominentes**: Dificultad de concentración es el síntoma más sobre-representado.
- **Combinación de internalización y externalización**: Síntomas depresivos junto con conductas antisociales.
- **Ideación suicida significativa**: A pesar de ser un grupo moderado en tamaño, tiene alta ideación suicida.
- **Conductas antisociales variadas**: Desde mentiras hasta crueldad animal y peleas.

### Clase 4 (k =6)

329 observaciones (39.6%)

  ---
  Variables distintivas:

     1. pmd012:
        SOBRE 1: 'SI' - 26% (86/329) vs 16% (129/831) global, +11pp
        SUB 1: 'NO' - 74% (243/329) vs 84% (702/831) global, -11pp
  

#### Perfil general:
La **Clase 4** muestra un **perfil de bajo riesgo o mínima psicopatología**, caracterizado por una **cercana similitud con la población general de la muestra en la mayoría de las variables**. Constituye el **grupo más grande (39.6%)** y representa el perfil más cercano a la norma dentro de esta población.

#### Hallazgos principales:

##### **Sintomatología depresiva leve:**
- **`pmd012`**: 26% reporta menos energía (vs 16% global) → **+11pp**

 Esta es la única variable con una diferencia igual o mayor a 10 puntos porcentuales (pp). Esto indica que, en comparación con las otras clases caracterizadas, la Clase 4 es notablemente similar al promedio global en prácticamente todos los aspectos medidos.



### Clase 5 (k =6) 
  318 observaciones (38.3%)

  ---
  Variables distintivas:

     1. pmd003:
        SOBRE 1: 'NO' - 80% (253/318) vs 56% (469/831) global, +23pp
        SUB 1: 'SI' - 20% (65/318) vs 44% (362/831) global, -23pp

     2. pmd018:
        SOBRE 1: 'NO' - 81% (256/318) vs 59% (489/831) global, +22pp
        SUB 1: 'SI' - 19% (62/318) vs 41% (342/831) global, -22pp

     3. pod007:
        SOBRE 1: 'NO' - 82% (261/318) vs 61% (511/831) global, +21pp
        SUB 1: 'SI' - 18% (57/318) vs 39% (320/831) global, -21pp

     4. pod008:
        SOBRE 1: 'NO' - 82% (262/318) vs 62% (516/831) global, +20pp
        SUB 1: 'SI' - 18% (56/318) vs 38% (315/831) global, -20pp

     5. pmd001:
        SOBRE 1: 'NO' - 84% (267/318) vs 66% (548/831) global, +18pp
        SUB 1: 'SI' - 16% (51/318) vs 34% (283/831) global, -18pp

     6. pmd015:
        SOBRE 1: 'NO' - 100% (318/318) vs 83% (691/831) global, +17pp

     7. pmd012:
        SOBRE 1: 'NO' - 100% (318/318) vs 84% (702/831) global, +16pp

     8. pmd007:
        SOBRE 1: 'NO' - 92% (294/318) vs 78% (646/831) global, +15pp
        SUB 1: 'SI' - 8% (24/318) vs 22% (185/831) global, -15pp

     9. pmd008:
        SOBRE 1: 'NO' - 90% (286/318) vs 76% (634/831) global, +14pp
        SUB 1: 'SI' - 10% (32/318) vs 24% (197/831) global, -14pp

    10. pmd013:
        SOBRE 1: 'NO' - 97% (308/318) vs 84% (698/831) global, +13pp

    11. pmd002:
        SOBRE 1: 'NO' - 96% (305/318) vs 83% (691/831) global, +13pp

    12. pmd006:
        SOBRE 1: 'NO' - 100% (318/318) vs 87% (726/831) global, +13pp

#### Perfil general:
La **Clase 5** muestra un **perfil de muy bajo riesgo o ausencia de psicopatología significativa**, caracterizado por una **notable ausencia de síntomas emocionales, conductuales y cognitivos**. Constituye el **segundo grupo más grande (38.3%)** y representa el perfil más saludable dentro de la muestra, con niveles de síntomas muy por debajo del promedio global.

#### Hallazgos principales:

##### **Ausencia marcada de irritabilidad y hostilidad:**
- **`pmd003`**: 80% NO está a menudo malhumorado e irritable (vs 56% global) → **+23pp**
- **`pod007`**: 82% NO está malhumorado/fácilmente irritable (vs 61% global) → **+21pp**
- **`pod008`**: 82% NO parece enojado con personas o cosas (vs 62% global) → **+20pp**

##### **Ausencia de problemas atencionales y cognitivos:**
- **`pmd018`**: 81% NO tiene dificultad para mantener la concentración (vs 59% global) → **+22pp**

##### **Ausencia de sintomatología depresiva:**
- **`pmd001`**: 84% NO ha tenido períodos de gran malestar/depresión (vs 66% global) → **+18pp**
- **`pmd002`**: 96% NO siente desinterés/tristeza (vs 83% global) → **+13pp**
- **`pmd015`**: 100% NO se culpa a sí mismo por cosas malas (vs 83% global) → **+17pp**

##### **Ausencia de síntomas neurovegetativos:**
- **`pmd012`**: 100% NO reporta menos energía (vs 84% global) → **+16pp**
- **`pmd013`**: 97% NO se siente fatigado (vs 84% global) → **+13pp**
- **`pmd007`**: 92% NO tiene aumento de apetito (vs 78% global) → **+15pp**
- **`pmd008`**: 90% NO tiene problemas para dormir (vs 76% global) → **+14pp**
- **`pmd006`**: 100% NO ha ganado mucho peso (vs 87% global) → **+13pp**

#### Variables más distintivas (diferencias >15pp):

| Variable | Descripción | % Clase 5 (NO) | % Global (NO) | Diferencia |
|----------|-------------|----------------|---------------|------------|
| **pmd003** | Sin irritabilidad/enojo por pequeñas cosas | 80% | 56% | **+23pp** |
| **pmd018** | Sin dificultad de concentración | 81% | 59% | **+22pp** |
| **pod007** | Sin mal humor/facilidad para irritarse | 82% | 61% | **+21pp** |
| **pod008** | Sin parecer enojado con personas/cosas | 82% | 62% | **+20pp** |
| **pmd001** | Sin períodos de gran malestar/depresión | 84% | 66% | **+18pp** |
| **pmd015** | Sin autoculpa | 100% | 83% | **+17pp** |
| **pmd012** | Sin menos energía | 100% | 84% | **+16pp** |

#### Síntesis interpretativa:

La **Clase 5** representa un **perfil de excelente salud mental o muy bajo riesgo psicológico** que se caracteriza por:

1. **Ausencia casi completa de irritabilidad**: Niveles muy bajos de mal humor, irritabilidad y enojo.
2. **Funcionamiento cognitivo preservado**: Sin problemas de concentración significativos.
3. **Ausencia de sintomatología depresiva**: Sin malestar depresivo, desinterés, tristeza o autoculpa.
4. **Sistema neurovegetativo estable**: Sin alteraciones en sueño, apetito, energía o peso.
5. **Estabilidad emocional general**: Baja prevalencia en prácticamente todos los síntomas evaluados.

Este perfil representa el **subgrupo más saludable** en la muestra, con niveles de síntomas notablemente bajos en comparación con el promedio global y con las otras clases. Su tamaño considerable (38.3%) sugiere que una proporción importante de la población estudiada presenta un ajuste psicológico muy bueno.

## Síntesis de Diferenciación de Clases (k=6)

| **Criterio** | **Clase 0 (Depresión Severa Mixta)** | **Clase 1 (Irritabilidad/Depresión Moderada)** | **Clase 2 (Externalización Grave Pura)** | **Clase 3 (Problemas Atencionales Mixtos)** | **Clase 4 (Bajo Riesgo/Normativa)** | **Clase 5 (Excelente Salud Mental)** |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Tamaño de la Clase** | 22 observaciones (2.6%) | 41 observaciones (4.9%) | 27 observaciones (3.2%) | 94 observaciones (11.3%) | 329 observaciones (39.6%) | 318 observaciones (38.3%) |
| **Patrón Principal** | **Depresión severa atípica con irritabilidad extrema y conductas externalizantes.** Combinación extrema de irritabilidad, paranoia e ideación suicida. | **Irritabilidad y depresión moderada con rasgos psicóticos y comportamientos externalizantes.** Síntomas emocionales, conductuales y cognitivos de intensidad moderada. | **Externalización grave con conductas antisociales múltiples y baja sintomatología internalizante.** Agresión intencional, delincuencia y hostilidad sin comorbilidad depresiva significativa. | **Perfil mixto con problemas atencionales prominentes, irritabilidad, conductas antisociales y síntomas depresivos.** Múltiples dominios afectados con dificultades cognitivas como núcleo. | **Baja psicopatología.** Cercana similitud con la población global, con solo un síntoma leve elevado (fatiga). | **Ausencia de psicopatología significativa.** Niveles de síntomas muy por debajo del promedio global en todos los dominios. |
| **Núcleo del Perfil** | **Síntomas depresivos con irritabilidad extrema.** Alteraciones severas en sueño, apetito, energía, peso, junto con hostilidad, paranoia y piromanía. | **Trastorno del estado de ánimo con irritabilidad prominente y características psicóticas incipientes.** Paranoia, irritabilidad, síntomas depresivos y bullying. | **Síntomas externalizantes con agresión instrumental.** Patrón diverso de conductas antisociales (hurto, robo, piromanía, crueldad animal) sin angustia emocional significativa. | **Síntomas relacionados a la falta de atención, irritabilidad e internalización.** Dificultades atencionales como síntoma principal, con irritabilidad, conductas antisociales e ideación suicida. | **Perfil normativo de referencia.** Representa la población base con mínimas desviaciones, actuando como grupo de comparación. | **Perfil resiliente o de alta salud mental.** Ausencia de síntomas en múltiples dominios, sugiriendo excelentes mecanismos de afrontamiento y factores protectores. |
| **Síntomas más Distintivos (Diferencia >20pp vs. Global)** | 1. **pmd008:** Problemas sueño (+72pp)<br>2. **pmd007:** Aumento apetito (+64pp)<br>3. **pod008:** Enojo (+53pp)<br>4. **pmd013:** Fatiga (+52pp)<br>5. **pmd003:** Irritabilidad (+52pp) | 1. **pod007:** Irritabilidad (+30pp)<br>2. **psz009:** Paranoia (+24pp)<br>3. **pod008:** Enojo (+23pp)<br>4. **pmd003:** Irritabilidad (+22pp)<br>5. **pmd018:** Dific. concentración (+22pp) | 1. **pod010:** Daño intencional (+42pp)<br>2. **pod008:** Enojo (+40pp)<br>3. **pcd003:** Robo (+37pp)<br>4. **pcd002:** Hurto (+32pp)<br>5. **pcd022:** Piromanía (+29pp) | 1. **pmd018:** Dific. concentración (+27pp)<br>2. **pmd003:** Irritabilidad (+23pp)<br>3. **pod008:** Enojo (+22pp)<br>4. **pcd023:** Crueldad animal (+20pp)<br>5. **pcd027:** Peleas (+19pp) | 1. **pmd012:** Menos energía (+11pp) | 1. **pmd003:** Sin irritabilidad (+23pp en 'NO')<br>2. **pmd018:** Sin dific. concentración (+22pp en 'NO')<br>3. **pod007:** Sin mal humor (+21pp en 'NO')<br>4. **pod008:** Sin enojo (+20pp en 'NO')<br>5. **pmd001:** Sin malestar (+18pp en 'NO') |
| **Riesgo Asociado** | **Muy Alto.** Riesgo de cronicidad depresiva, suicidio, exacerbación de síntomas psicóticos y conductas de riesgo (piromanía). | **Moderado-Alto.** Riesgo de evolución a trastorno psicótico, escalada de conductas externalizantes y exacerbación de síntomas depresivos. | **Alto.** Riesgo de persistencia de conducta antisocial, problemas legales, y escalada de violencia. Baja percepción de angustia puede dificultar la motivación para el cambio. | **Moderado-Alto.** Riesgo de fracaso académico por problemas atencionales, cronificación de conductas antisociales y exacerbación de ideación suicida. | **Bajo.** Representa el perfil de ajuste normal. El riesgo principal es la posible transición a clases problemáticas sin intervenciones preventivas. | **Mínimo.** Representa el perfil de salud mental óptima. Puede servir como modelo de factores protectores y resiliencia. |
| **Abordaje Sugerido** | **Tratamiento intensivo.** Manejo farmacológico especializado (antidepresivos, estabilizadores), psicoterapia intensiva para depresión y control de impulsos, evaluación constante de riesgo suicida y de piromanía. | **Tratamiento ambulatorio especializado.** Evaluación psiquiátrica para síntomas psicóticos, terapia cognitivo-conductual para irritabilidad y paranoia, intervenciones para bullying y manejo de conducta. | **Intervenciones conductuales especializadas y posible marco judicial.** Terapia cognitivo-conductual centrada en conductas antisociales, programas de responsabilidad, manejo de la ira, y evaluación de riesgo por piromanía y crueldad animal. | **Tratamiento multimodal integrado.** Evaluación neuropsicológica para TDAH, combinación de intervenciones farmacológicas (estimulantes, antidepresivos) y psicosociales, apoyo escolar, manejo de riesgo suicida. | **Prevención universal y psicoeducación.** Programas de promoción de salud mental, manejo del estrés y hábitos de sueño. Detección temprana de síntomas emergentes. | **Promoción de la salud mental y estudio de factores protectores.** Identificación y replicación de factores de resiliencia, programas de mentoría donde puedan ser modelos positivos, mantenimiento del bienestar. |

---

# Clasificación LCA para Jóvenes

## Variables más discriminatorias (k = 3)

| Variable | Cramér's V | p-value   | χ²  | Discriminación |
|----------|------------|-----------|-----|----------------|
| pcd019   | 0.4943     | 1.11e-73  | 355 | 0.969          |
| pmd022   | 0.3775     | 5.48e-42  | 207 | 0.978          |
| pmd021   | 0.3634     | 9.54e-39  | 192 | 0.978          |
| pmd015   | 0.3260     | 8.49e-31  | 155 | 0.978          |
| pcd008   | 0.3249     | 1.40e-30  | 154 | 0.969          |
| pcd009   | 0.3249     | 1.40e-30  | 154 | 0.969          |
| psu011   | 0.3249     | 1.40e-30  | 154 | 0.969          |
| psu003   | 0.3234     | 2.82e-30  | 152 | 0.969          |
| psz011   | 0.3117     | 5.39e-28  | 141 | 0.978          |
| psz012   | 0.3092     | 1.60e-27  | 139 | 0.978          |
| psu001   | 0.3006     | 6.53e-26  | 131 | 0.969          |
| pcd007   | 0.3006     | 6.53e-26  | 131 | 0.969          |
| psu002   | 0.3006     | 6.53e-26  | 131 | 0.969          |
| pcd029   | 0.2939     | 3.43e-26  | 126 | 0.979          |
| pcd020   | 0.2892     | 7.47e-24  | 122 | 0.969          |
| pcd027   | 0.2838     | 6.49e-23  | 117 | 0.969          |
| pad033   | 0.2809     | 2.08e-22  | 115 | 0.969          |
| pod001   | 0.2809     | 2.09e-22  | 115 | 0.969          |
| pcd028   | 0.2752     | 1.92e-21  | 110 | 0.969          |
| psu009   | 0.2742     | 2.79e-21  | 109 | 0.969          |
| pcd012   | 0.2725     | 5.48e-21  | 108 | 0.969          |
| pmd020   | 0.2690     | 2.03e-20  | 105 | 0.978          |
| pmd010   | 0.2690     | 2.05e-20  | 105 | 0.978          |
| pmd035   | 0.2670     | 1.62e-21  | 104 | 0.880          |
| psz010   | 0.2670     | 4.29e-20  | 104 | 0.978          |

## Hallazgos principales:

### Variable más discriminante:
- **`pcd019`** – *"Ever broken into a house/building/car"*
    - **Cramér's V = 0.4943** (el más alto)
    - **p = 1.11e-73**

### Dimensiones temáticas predominantes:

#### **Conducta antisocial y delictiva grave:**
- `pcd019` – Allanamiento de vivienda/edificio/auto
- `pcd008` – Asaltar/atacar para robar
- `pcd009` – Amenazar para robar
- `pcd007` – Robar bolso o joyería
- `pcd029` – Herir a alguien con un arma
- `pcd027` – Participar en peleas físicas con lesiones
- `pcd028` – Intentar lastimar gravemente a alguien
- `pcd020` – Destruir propiedad intencionalmente
- `pad033` – Estar en una pelea
- `pcd012` – Problemas por incumplir toque de queda

#### **Ideación y conducta suicida:**
- `pmd022` – Intento de suicidio
- `pmd021` – Hablar seriamente de suicidarse
- `pmd020` – Pensamientos sobre muerte/morir

#### **Síntomas depresivos:**
- `pmd015` – Culparse por cosas malas que sucedieron
- `pmd010` – Lentitud psicomotora (hacer cosas más lento)
- `pmd035` – Aparecer triste o deprimido mucho tiempo

#### **Consumo de sustancias ilegales:**
- `psu011` – Uso de otras drogas para drogarse
- `psu003` – Uso de cocaína o "crack"
- `psu001` – Uso de estimulantes o anfetaminas
- `psu002` – Uso de sedantes o tranquilizantes
- `psu009` – Uso de inhalantes

#### **Síntomas psicóticos (paranoia/ideas delirantes):**
- `psz011` – Creer que alguien conspiraba contra ti/trataba de lastimarte
- `psz012` – Convencido de que pensamientos extraños fueron puestos en tu mente
- `psz010` – Pensamientos de gente hablando/riéndose de ti

#### **Comportamiento agresivo y oposicionista:**
- `pod001` – Perder el temperamento

## Variables más discriminatorias (k = 3)

### Clase 0 (k = 3)

 471 observaciones (64.8%)
  
  ---
  Variables distintivas:

     1. pmd015:
        SOBRE 1: 'NO' - 80% (379/471) vs 66% (480/727) global, +14pp
        SUB 1: 'SI' - 20% (92/471) vs 34% (244/727) global, -14pp

     2. pmd035:
        SOBRE 1: 'NO' - 82% (386/471) vs 70% (506/727) global, +12pp

     3. pmd022:
        SOBRE 1: 'NO' - 100% (471/471) vs 88% (641/727) global, +12pp

     4. pmd020:
        SOBRE 1: 'NO' - 76% (357/471) vs 65% (471/727) global, +11pp
        SUB 1: 'SI' - 24% (114/471) vs 35% (253/727) global, -11pp

     5. psz010:
        SOBRE 1: 'NO' - 77% (363/471) vs 66% (482/727) global, +11pp
        SUB 1: 'SI' - 23% (108/471) vs 33% (242/727) global, -10pp

     6. pmd021:
        SOBRE 1: 'NO' - 100% (471/471) vs 89% (649/727) global, +11pp

#### Perfil general:
La **Clase 0** es el grupo más grande con el **64.8%** de la muestra (471/727 observaciones). Muestra un **perfil de baja sintomatología internalizante y psicótica**, caracterizado por una menor prevalencia de síntomas depresivos, suicidas y psicóticos en comparación con la muestra global.

#### Hallazgos principales:

##### **Baja sintomatología depresiva y suicida:**
- **`pmd015`**:  **20%** se culpa por cosas malas que sucedieron (vs 34% global) → **-14pp**
- **`pmd022`**: **0%** ha intentado suicidarse (vs 12% global) → **-12pp** (sobre representación de 'NO': 100% vs 88% global)
- **`pmd021`**: **0%** ha hablado seriamente de suicidarse (vs 11% global) → **-11pp** (sobre representación de 'NO': 100% vs 89% global)
- **`pmd020`**:  **24%** tiene pensamientos sobre muerte/morir (vs 35% global) → **-11pp**
- **`pmd035`**:  **18%** aparece triste/deprimido mucho tiempo (vs 30% global) → **-12pp** (sobre representación de 'NO': 82% vs 70% global)

##### **Baja sintomatología psicótica:**
- **`psz010`**: Solo el **23%** tiene pensamientos de gente hablando/riéndose de él/ella (vs 33% global) → **-10pp**

#### Variables más distintivas (mayores diferencias):

| Variable | Descripción | % Clase 0 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pmd015** | Culparse por cosas malas  | 20% | 34% | **-14pp** |
| **pmd022** | Intento de suicidio  | 0% | 12% | **-12pp** |
| **pmd035** | Aparecer triste/deprimido  | 18% | 30% | **-12pp** |
| **pmd021** | Hablar seriamente de suicidarse  | 0% | 11% | **-11pp** |
| **pmd020** | Pensamientos sobre muerte  | 24% | 35% | **-11pp** |
| **psz010** | Pensamientos de gente hablando/riéndose  | 23% | 33% | **-10pp** |

#### Síntesis interpretativa:

La **Clase 0** representa el **grupo de referencia o de menor gravedad** dentro de la muestra, que se caracteriza por:

1. **Baja prevalencia de síntomas internalizantes graves**: especialmente ausencia total de ideación y conducta suicida.
2. **Baja prevalencia de síntomas psicóticos**: menos ideas de referencia y paranoia.
3. **Menor tendencia a la autorreproche y culpa**.
4. **Menor afectación del estado de ánimo** depresivo sostenido.


Este patrón sugiere un **subgrupo que presenta menos riesgo clínico inmediato** y que podría representar:
- Casos de menor severidad dentro de la muestra clínica
- Individuos con mayor resiliencia o recursos adaptativos
- Posiblemente un grupo que requiere intervenciones menos intensivas



### Clase 1 (k = 3)
  224 observaciones (30.8%)
  
  ---
  Variables distintivas:

     1. pmd015:
        SOBRE 1: 'SI' - 58% (131/224) vs 34% (244/727) global, +25pp
        SUB 1: 'NO' - 41% (92/224) vs 66% (480/727) global, -25pp

     2. pmd035:
        SOBRE 1: 'SI' - 36% (80/224) vs 22% (163/727) global, +13pp
        SUB 1: 'NO' - 47% (105/224) vs 70% (506/727) global, -23pp

     3. psz010:
        SOBRE 1: 'SI' - 55% (123/224) vs 33% (242/727) global, +22pp
        SUB 1: 'NO' - 45% (100/224) vs 66% (482/727) global, -22pp

     4. pmd022:
        SOBRE 1: 'SI' - 33% (74/224) vs 11% (83/727) global, +22pp
        SUB 1: 'NO' - 67% (149/224) vs 88% (641/727) global, -22pp

     5. pmd021:
        SOBRE 1: 'SI' - 31% (69/224) vs 10% (75/727) global, +20pp
        SUB 1: 'NO' - 69% (154/224) vs 89% (649/727) global, -21pp

     6. pmd010:
        SOBRE 1: 'SI' - 40% (89/224) vs 21% (154/727) global, +19pp
        SUB 1: 'NO' - 60% (134/224) vs 78% (570/727) global, -19pp

     7. pmd020:
        SOBRE 1: 'SI' - 52% (117/224) vs 35% (253/727) global, +17pp
        SUB 1: 'NO' - 47% (106/224) vs 65% (471/727) global, -17pp

     8. pod001:
        SOBRE 1: 'SI' - 50% (112/224) vs 35% (258/727) global, +15pp
        SUB 1: 'NO' - 50% (112/224) vs 64% (466/727) global, -14pp

     9. psz011:
        SOBRE 1: 'SI' - 21% (48/224) vs 7% (54/727) global, +14pp
        SUB 1: 'NO' - 78% (175/224) vs 92% (670/727) global, -14pp

    10. psz012:
        SOBRE 1: 'SI' - 21% (46/224) vs 7% (53/727) global, +13pp
        SUB 1: 'NO' - 79% (177/224) vs 92% (671/727) global, -13pp

    11. pad033:
        SOBRE 1: 'SI' - 32% (71/224) vs 21% (156/727) global, +10pp

#### Perfil general:
La **Clase 1** es un grupo de tamaño mediano con el **30.8%** de la muestra (224/727 observaciones). Muestra un **perfil de alta sintomatología internalizante, psicótica y depresiva grave**, caracterizado por una mayor prevalencia de síntomas depresivos severos, conducta suicida, síntomas psicóticos y agresividad.

#### Hallazgos principales:

##### **Alta sintomatología depresiva y suicida:**
- **`pmd015`**:  **58%** se culpa por cosas malas que sucedieron (vs 34% global) → **+25pp**
- **`pmd022`**: **33%** ha intentado suicidarse (vs 11% global) → **+22pp**
- **`pmd021`**: **31%** ha hablado seriamente de suicidarse (vs 10% global) → **+20pp**
- **`pmd010`**: **40%** muestra lentitud psicomotora (vs 21% global) → **+19pp**
- **`pmd020`**: **52%** tiene pensamientos sobre muerte/morir (vs 35% global) → **+17pp**
- **`pmd035`**: **36%** aparece triste/deprimido mucho tiempo (vs 22% global) → **+13pp**

##### **Alta sintomatología psicótica:**
- **`psz010`**: **55%** tiene pensamientos de gente hablando/riéndose de él/ella (vs 33% global) → **+22pp**
- **`psz011`**: **21%** cree que alguien conspiraba contra él/ella (vs 7% global) → **+14pp**
- **`psz012`**: **21%** está convencido de que pensamientos extraños fueron puestos en su mente (vs 7% global) → **+13pp**

##### **Comportamiento agresivo y disruptivo:**
- **`pod001`**:  **50%** pierde el temperamento (vs 35% global) → **+15pp**
- **`pad033`**:  **32%** ha estado en una pelea (vs 21% global) → **+10pp**

#### Variables más distintivas (diferencias >15pp):

| Variable | Descripción | % Clase 1 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pmd015** | Culparse por cosas malas  | 58% | 34% | **+25pp** |
| **psz010** | Pensamientos de gente hablando/riéndose  | 55% | 33% | **+22pp** |
| **pmd022** | Intento de suicidio  | 33% | 11% | **+22pp** |
| **pmd021** | Hablar seriamente de suicidarse  | 31% | 10% | **+20pp** |
| **pmd010** | Lentitud psicomotora  | 40% | 21% | **+19pp** |
| **pmd020** | Pensamientos sobre muerte  | 52% | 35% | **+17pp** |

#### Síntesis interpretativa:

La **Clase 1** representa un **grupo de alta gravedad clínica** que se caracteriza por:

1. **Sintomatología depresiva grave**: con altos niveles de culpa, pensamientos de muerte e ideación suicida.
2. **Conducta suicida significativa**: un tercio ha intentado suicidarse y casi un tercio ha hablado seriamente de ello.
3. **Síntomas psicóticos prominentes**: ideas de referencia, paranoia y experiencias de inserción de pensamientos.
4. **Agresividad y pérdida de control emocional**: alta irritabilidad y participación en peleas.

Es importante notar que, aunque hay comportamiento agresivo, no es el foco principal; el foco está en la sintomatología internalizante y psicótica grave, lo que diferencia esta clase de aquellas centradas en conducta antisocial pura.
  

### Clase 2 (k = 3)
  32 observaciones (4.4%)
  
  ---
  Variables distintivas:

     1. pcd027:
        SOBRE 1: 'SI' - 78% (25/32) vs 27% (199/727) global, +51pp
        SUB 1: 'NO' - 12% (4/32) vs 72% (525/727) global, -60pp

     2. pcd012:
        SOBRE 1: 'SI' - 59% (19/32) vs 20% (143/727) global, +40pp
        SUB 1: 'NO' - 31% (10/32) vs 80% (581/727) global, -49pp

     3. pcd020:
        SOBRE 1: 'SI' - 50% (16/32) vs 11% (81/727) global, +39pp
        SUB 1: 'NO' - 41% (13/32) vs 88% (643/727) global, -48pp

     4. pcd019:
        SOBRE 1: 'SI' - 41% (13/32) vs 2% (13/727) global, +39pp
        SUB 1: 'NO' - 50% (16/32) vs 98% (711/727) global, -48pp

     5. pad033:
        SOBRE 1: 'SI' - 53% (17/32) vs 21% (156/727) global, +32pp
        SUB 1: 'NO' - 38% (12/32) vs 78% (568/727) global, -41pp

     6. pmd020:
        SOBRE 1: 'SI' - 69% (22/32) vs 35% (253/727) global, +34pp
        SUB 1: 'NO' - 25% (8/32) vs 65% (471/727) global, -40pp

     7. pmd015:
        SOBRE 1: 'SI' - 66% (21/32) vs 34% (244/727) global, +32pp
        SUB 1: 'NO' - 28% (9/32) vs 66% (480/727) global, -38pp

     8. pcd028:
        SOBRE 1: 'SI' - 34% (11/32) vs 7% (48/727) global, +28pp
        SUB 1: 'NO' - 56% (18/32) vs 93% (676/727) global, -37pp

     9. pod001:
        SOBRE 1: 'SI' - 62% (20/32) vs 35% (258/727) global, +27pp
        SUB 1: 'NO' - 28% (9/32) vs 64% (466/727) global, -36pp

    10. pcd029:
        SOBRE 1: 'SI' - 31% (10/32) vs 4% (26/727) global, +28pp
        SUB 1: 'NO' - 62% (20/32) vs 96% (699/727) global, -34pp

    11. psu003:
        SOBRE 1: 'SI' - 22% (7/32) vs 2% (12/727) global, +20pp
        SUB 1: 'NO' - 69% (22/32) vs 98% (712/727) global, -29pp

    12. pmd035:
        SOBRE 1: 'SI' - 38% (12/32) vs 22% (163/727) global, +15pp
        SUB 1: 'NO' - 47% (15/32) vs 70% (506/727) global, -23pp

    13. pmd022:
        SOBRE 1: 'SI' - 28% (9/32) vs 11% (83/727) global, +17pp
        SUB 1: 'NO' - 66% (21/32) vs 88% (641/727) global, -23pp

    14. pcd008:
        SOBRE 1: 'SI' - 12% (4/32) vs 1% (4/727) global, +12pp
        SUB 1: 'NO' - 78% (25/32) vs 99% (720/727) global, -21pp

    15. pcd009:
        SOBRE 1: 'SI' - 12% (4/32) vs 1% (4/727) global, +12pp
        SUB 1: 'NO' - 78% (25/32) vs 99% (720/727) global, -21pp

    16. psu011:
        SOBRE 1: 'SI' - 12% (4/32) vs 1% (4/727) global, +12pp
        SUB 1: 'NO' - 78% (25/32) vs 99% (720/727) global, -21pp

    17. psz012:
        SOBRE 1: 'SI' - 22% (7/32) vs 7% (53/727) global, +15pp
        SUB 1: 'NO' - 72% (23/32) vs 92% (671/727) global, -20pp

    18. psu001:
        SUB 1: 'NO' - 81% (26/32) vs 99% (721/727) global, -18pp

    19. pcd007:
        SUB 1: 'NO' - 81% (26/32) vs 99% (721/727) global, -18pp

    20. psu002:
        SUB 1: 'NO' - 81% (26/32) vs 99% (721/727) global, -18pp

    21. psz011:
        SOBRE 1: 'SI' - 19% (6/32) vs 7% (54/727) global, +11pp
        SUB 1: 'NO' - 75% (24/32) vs 92% (670/727) global, -17pp

    22. psu009:
        SUB 1: 'NO' - 84% (27/32) vs 99% (722/727) global, -15pp

    23. pmd021:
        SUB 1: 'NO' - 75% (24/32) vs 89% (649/727) global, -14pp

    24. pmd010:
        SUB 1: 'NO' - 66% (21/32) vs 78% (570/727) global, -13pp

#### Perfil general:
La **Clase 2** es el grupo más pequeño pero más extremo con solo el **4.4%** de la muestra (32/727 observaciones). Muestra un **perfil de externalización grave con conducta antisocial violenta, delitos graves y consumo de drogas duras**, caracterizado por una prevalencia extremadamente alta de conductas delictivas violentas junto con sintomatología depresiva.

#### Hallajzos principales:

##### **Conducta violenta extrema:**
- **`pcd027`**: **78%** ha participado en peleas físicas con lesiones (vs 27% global) → **+51pp**
- **`pcd019`**:  **41%** ha cometido allanamiento de vivienda/edificio/auto (vs 2% global) → **+39pp**
- **`pcd020`**:  **50%** ha destruido propiedad intencionalmente (vs 11% global) → **+39pp**
- **`pcd029`**:  **31%** ha herido a alguien con un arma (vs 4% global) → **+28pp**
- **`pcd028`**:  **34%** ha intentado lastimar gravemente a alguien (vs 7% global) → **+28pp**
- **`pad033`**:  **53%** ha estado en una pelea (vs 21% global) → **+32pp**

##### **Transgresiones graves y problemas conductuales:**
- **`pcd012`**:  **59%** ha tenido problemas por incumplir toque de queda (vs 20% global) → **+40pp**
- **`pod001`**:  **62%** pierde el temperamento (vs 35% global) → **+27pp**

##### **Consumo de sustancias duras:**
- **`psu003`**: **22%** ha usado cocaína o "crack" (vs 2% global) → **+20pp**
- **`psu011`**:  **12%** ha usado otras drogas para drogarse (vs 1% global) → **+12pp**

##### **Sintomatología depresiva y suicida:**
- **`pmd020`**: **69%** tiene pensamientos sobre muerte/morir (vs 35% global) → **+34pp**
- **`pmd015`**: **66%** se culpa por cosas malas que sucedieron (vs 34% global) → **+32pp**
- **`pmd022`**:  **28%** ha intentado suicidarse (vs 11% global) → **+17pp**
- **`pmd035`**:  **38%** aparece triste/deprimido mucho tiempo (vs 22% global) → **+15pp**

#### Variables más distintivas (diferencias >30pp):

| Variable | Descripción | % Clase 2 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pcd027** | Peleas físicas con lesiones  | 78% | 27% | **+51pp** |
| **pcd012** | Problemas por incumplir toque de queda  | 59% | 20% | **+40pp** |
| **pcd020** | Destruir propiedad intencionalmente  | 50% | 11% | **+39pp** |
| **pcd019** | Allanamiento de vivienda/edificio/auto  | 41% | 2% | **+39pp** |
| **pad033** | Estar en una pelea  | 53% | 21% | **+32pp** |
| **pmd020** | Pensamientos sobre muerte  | 69% | 35% | **+34pp** |
| **pmd015** | Culparse por cosas malas  | 66% | 34% | **+32pp** |

#### Síntesis interpretativa:

La **Clase 2** representa un **grupo de extremada gravedad y alto riesgo** que se caracteriza por:

1. **Conducta antisocial violenta extrema**: altas tasas de agresión física, vandalismo grave y delitos contra la propiedad.
2. **Patrón delictivo diversificado**: incluye violencia interpersonal, daño a propiedad y transgresiones graves.
3. **Consumo de sustancias duras**: notable uso de cocaína/crack y otras drogas.
4. **Comorbilidad con sintomatología depresiva grave**: pensamientos de muerte, culpa excesiva y conducta suicida.

Esta clase representa probablemente el **subgrupo de mayor complejidad y cronicidad** en la muestra, con necesidades que exceden los servicios de salud mental tradicionales y requieren abordajes multisistémicos.

## Síntesis de Diferenciación de Clases (k=3)

| **Criterio** | **Clase 0 (Bajo Riesgo)** | **Clase 1 (Internalización Grave)** | **Clase 2 (Externalización Grave)** |
| :--- | :--- | :--- | :--- |
| **Tamaño de la Clase** | 471 observaciones (64.8%) | 224 observaciones (30.8%) | 32 observaciones (4.4%) |
| **Patrón Principal** | **Baja sintomatología internalizante y psicótica.** Ausencia o baja frecuencia de síntomas depresivos, suicidas y psicóticos en comparación con la muestra global. | **Alta sintomatología internalizante, psicótica y depresiva grave.** Presencia marcada de ideación e intentos suicidas, síntomas psicóticos y alteraciones severas del estado de ánimo. | **Conducta antisocial violenta extrema con consumo de drogas y sintomatología depresiva.** Patrón de externalización grave con delitos violentos, vandalismo, consumo de sustancias duras y comorbilidad depresiva. |
| **Núcleo del Perfil** | **Ausencia de psicopatología significativa.** Se caracteriza por la **sub-representación** de síntomas problemáticos, actuando como grupo de referencia normativo dentro de la muestra clínica. | **Depresión grave con síntomas psicóticos y riesgo suicida elevado.** Combinación de culpa excesiva, ideación e intentos suicidas, lentitud psicomotora y experiencias psicóticas de tipo paranoide (ideas de referencia, persecución). | **Tríada de violencia, transgresiones graves y depresión.** Combinación extrema de conductas violentas (peleas con lesiones, uso de armas, allanamiento), problemas de conducta disruptiva (incumplir toque de queda) y depresión con pensamientos de muerte y culpa. |
| **Síntomas más Distintivos (Diferencia >15pp vs. Global)** | 1. **pmd015:** Baja culpa (-14pp)<br>2. **pmd022:** Sin intentos suicidas (-12pp)<br>3. **pmd035:** Baja tristeza (-12pp)<br>4. **pmd021:** Sin ideación suicida grave (-11pp)<br>5. **pmd020:** Bajos pensamientos de muerte (-11pp) | 1. **pmd015:** Alta culpa (+25pp)<br>2. **psz010:** Pensamientos de referencia (+22pp)<br>3. **pmd022:** Intentos suicidas (+22pp)<br>4. **pmd021:** Ideación suicida (+20pp)<br>5. **pmd010:** Lentitud psicomotora (+19pp)<br>6. **pmd020:** Pensamientos de muerte (+17pp) | 1. **pcd027:** Peleas con lesiones (+51pp)<br>2. **pcd012:** Problemas por toque de queda (+40pp)<br>3. **pcd020:** Destrucción propiedad (+39pp)<br>4. **pcd019:** Allanamiento (+39pp)<br>5. **pmd020:** Pensamientos de muerte (+34pp)<br>6. **pad033:** Peleas (+32pp)<br>7. **pmd015:** Culpa (+32pp) |
| **Riesgo Asociado** | **Muy Bajo.** Riesgo clínico mínimo, representando el perfil de menor gravedad en la muestra. El riesgo principal es la posible transición hacia clases problemáticas sin intervenciones preventivas. | **Alto.** Riesgo suicida elevado (ideación e intentos), posible trastorno depresivo mayor con características psicóticas, y deterioro funcional significativo. | **Muy Alto.** Riesgo extremo de violencia hacia otros y hacia sí mismo (suicidio), implicación con el sistema judicial, cronicidad y complicaciones graves por consumo de sustancias. |
| **Abordaje Sugerido** | **Prevención universal y promoción de la salud mental.** Monitoreo de síntomas emergentes, fortalecimiento de factores protectores y educación psico-social para mantener el ajuste adaptativo. | **Intervención intensiva y especializada.** Evaluación psiquiátrica urgente, posible hospitalización para manejo de riesgo suicida, tratamiento farmacológico (antidepresivos y antipsicóticos) y psicoterapia para síntomas depresivos y psicóticos. | **Tratamiento multidisciplinario intensivo y manejo de justicia.** Abordaje integral que combine: intervenciones para la conducta violenta (programas de control de impulsos), tratamiento de adicciones a sustancias duras, manejo de la depresión con riesgo suicida, y coordinación con sistemas de justicia juvenil. |

## Variables más discriminatorias (k = 6)

### Clase 0 (k = 6)

 78 observaciones (10.7%)
  
  ---
  Variables distintivas:

     1. pad033:
        SOBRE 1: 'SI' - 55% (43/78) vs 21% (156/727) global, +34pp
        SUB 1: 'NO' - 45% (35/78) vs 78% (568/727) global, -33pp

     2. pad025:
        SOBRE 1: 'SI' - 40% (31/78) vs 11% (83/727) global, +28pp
        SUB 1: 'NO' - 60% (47/78) vs 88% (641/727) global, -28pp

     3. pod005:
        SOBRE 1: 'SI' - 54% (42/78) vs 28% (202/727) global, +26pp
        SUB 1: 'NO' - 46% (36/78) vs 72% (522/727) global, -26pp

     4. pod003:
        SOBRE 1: 'SI' - 56% (44/78) vs 32% (230/727) global, +25pp
        SUB 1: 'NO' - 44% (34/78) vs 68% (494/727) global, -24pp

     5. pod012:
        SOBRE 1: 'SI' - 44% (34/78) vs 20% (145/727) global, +24pp
        SUB 1: 'NO' - 56% (44/78) vs 80% (579/727) global, -23pp

     6. pcd027:
        SOBRE 1: 'SI' - 49% (38/78) vs 27% (199/727) global, +21pp
        SUB 1: 'NO' - 51% (40/78) vs 72% (525/727) global, -21pp

     7. pod011:
        SOBRE 1: 'SI' - 28% (22/78) vs 8% (57/727) global, +20pp
        SUB 1: 'NO' - 72% (56/78) vs 92% (667/727) global, -20pp

     8. pcd022:
        SOBRE 1: 'SI' - 38% (30/78) vs 19% (136/727) global, +20pp
        SUB 1: 'NO' - 62% (48/78) vs 81% (588/727) global, -19pp

     9. pcd021:
        SOBRE 1: 'SI' - 29% (23/78) vs 11% (78/727) global, +19pp
        SUB 1: 'NO' - 71% (55/78) vs 89% (646/727) global, -18pp

    10. pcd020:
        SOBRE 1: 'SI' - 28% (22/78) vs 11% (81/727) global, +17pp
        SUB 1: 'NO' - 72% (56/78) vs 88% (643/727) global, -17pp

    11. pmd022:
        SOBRE 1: 'SI' - 28% (22/78) vs 11% (83/727) global, +17pp
        SUB 1: 'NO' - 72% (56/78) vs 88% (641/727) global, -16pp

    12. pod010:
        SOBRE 1: 'SI' - 28% (22/78) vs 12% (86/727) global, +16pp
        SUB 1: 'NO' - 72% (56/78) vs 88% (638/727) global, -16pp

    13. pni001:
        SOBRE 1: 'SI' - 53% (41/78) vs 42% (307/727) global, +10pp
  
#### Perfil general:
La **Clase 0** es un grupo pequeño que representa el **10.7%** de la muestra (78/727 observaciones). Muestra un **perfil de externalización grave con conducta disruptiva, agresiva y transgresora**, caracterizado por una alta prevalencia de problemas de conducta oposicionista, agresión física, vandalismo, intentos de suicidio y tabaquismo.

#### Hallazgos principales:

##### **Conducta disruptiva y oposicionista:**
- **`pod003`**: **56%** ha hecho cosas que sus cuidadores le prohibieron a propósito (vs 32% global) → **+25pp**
- **`pod005`**: **54%** hace cosas para molestar/enojar a otros (vs 28% global) → **+26pp**
- **`pod012`**: **44%** ha insultado/usado lenguaje obsceno (vs 20% global) → **+24pp**
- **`pad025`**: **40%** ha trepado/corrido cuando no debía (vs 11% global) → **+28pp**

##### **Agresión física y peleas:**
- **`pad033`**: **55%** ha estado en una pelea (vs 21% global) → **+34pp**
- **`pcd027`**: **49%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **+21pp**

##### **Vandalismo y daño a propiedad:**
- **`pcd022`**: **38%** ha iniciado un fuego sin permiso (vs 19% global) → **+20pp**
- **`pcd021`**: **29%** ha dañado las cosas de otros a propósito (vs 11% global) → **+19pp**
- **`pcd020`**: **28%** ha destrozado un lugar intencionalmente (vs 11% global) → **+17pp**

##### **Conducta vengativa y mezquina:**
- **`pod011`**: **28%** se ha vengado de personas dañando sus cosas/hiriéndolas (vs 8% global) → **+20pp**
- **`pod010`**: **28%** ha hecho cosas malas a personas intencionalmente (vs 12% global) → **+16pp**

##### **Intento de suicidio y tabaquismo:**
- **`pmd022`**: **28%** ha intentado suicidarse (vs 11% global) → **+17pp**
- **`pni001`**: **53%** ha fumado cigarrillos (vs 42% global) → **+10pp**

#### Variables más distintivas (diferencias >20pp):

| Variable | Descripción | % Clase 0 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pad033** | Estar en una pelea  | 55% | 21% | **+34pp** |
| **pad025** | Trepar/correr cuando no debía  | 40% | 11% | **+28pp** |
| **pod005** | Molestar a otros intencionalmente  | 54% | 28% | **+26pp** |
| **pod003** | Desobediencia intencional  | 56% | 32% | **+25pp** |
| **pod012** | Insultar/lenguaje obsceno  | 44% | 20% | **+24pp** |
| **pcd027** | Peleas físicas con lesiones  | 49% | 27% | **+21pp** |
| **pod011** | Conducta vengativa  | 28% | 8% | **+20pp** |
| **pcd022** | Iniciar fuego sin permiso  | 38% | 19% | **+20pp** |

#### Síntesis interpretativa:

La **Clase 0** representa un **grupo de externalización con múltiples problemas conductuales y agresivos** que se caracteriza por:

1. **Conducta disruptiva y oposicionista marcada**: alta desobediencia intencional, conductas para molestar a otros y uso de lenguaje obsceno.
2. **Agresión física frecuente**: participación en peleas, con un alto porcentaje que ha estado en peleas con lesiones.
3. **Vandalismo y daño a propiedad**: desde destrozar lugares hasta iniciar fuegos sin permiso.
4. **Conducta vengativa y mezquina**: acciones para dañar a otros intencionalmente.
5. **Intento de suicidio y tabaquismo**: un subgrupo con intentos suicidas y consumo de tabaco.

### Clase 1 (k = 6)
  338 observaciones (46.5%)
  
  ---
  Variables distintivas:

     1. pod003:
        SOBRE 1: 'NO' - 86% (289/338) vs 68% (494/727) global, +18pp
        SUB 1: 'SI' - 14% (49/338) vs 32% (230/727) global, -17pp

     2. pcd027:
        SOBRE 1: 'NO' - 88% (298/338) vs 72% (525/727) global, +16pp
        SUB 1: 'SI' - 12% (40/338) vs 27% (199/727) global, -16pp

     3. pad033:
        SOBRE 1: 'NO' - 93% (316/338) vs 78% (568/727) global, +15pp
        SUB 1: 'SI' - 7% (22/338) vs 21% (156/727) global, -15pp

     4. pod005:
        SOBRE 1: 'NO' - 87% (293/338) vs 72% (522/727) global, +15pp
        SUB 1: 'SI' - 13% (45/338) vs 28% (202/727) global, -14pp

     5. pni001:
        SOBRE 1: 'NO' - 72% (244/338) vs 57% (417/727) global, +15pp
        SUB 1: 'SI' - 28% (94/338) vs 42% (307/727) global, -14pp

     6. pod010:
        SOBRE 1: 'NO' - 100% (338/338) vs 88% (638/727) global, +12pp

     7. pad025:
        SOBRE 1: 'NO' - 100% (338/338) vs 88% (641/727) global, +12pp

     8. pmd022:
        SOBRE 1: 'NO' - 100% (338/338) vs 88% (641/727) global, +12pp

     9. pcd022:
        SOBRE 1: 'NO' - 93% (313/338) vs 81% (588/727) global, +12pp
        SUB 1: 'SI' - 7% (25/338) vs 19% (136/727) global, -11pp

    10. pcd020:
        SOBRE 1: 'NO' - 100% (338/338) vs 88% (643/727) global, +12pp

    11. pcd021:
        SOBRE 1: 'NO' - 100% (338/338) vs 89% (646/727) global, +11pp

    12. pod012:
        SOBRE 1: 'NO' - 90% (304/338) vs 80% (579/727) global, +10pp

#### Perfil general:
La **Clase 1**  con el **46.5%** de la muestra (338/727 observaciones) presenta un **perfil de baja sintomatología con mínimos problemas conductuales y emocionales**, caracterizado por una marcada ausencia de conductas disruptivas, agresivas, transgresoras y de riesgo en comparación con la muestra global.

#### Hallazgos principales:

##### **Baja conducta disruptiva y oposicionista:**
- **`pod003`**: Solo el **14%** ha hecho cosas que sus cuidadores le prohibieron a propósito (vs 32% global) → **-17pp** (sobre: 'NO' +18pp)
- **`pod005`**: Solo el **13%** hace cosas para molestar/enojar a otros (vs 28% global) → **-14pp** (sobre: 'NO' +15pp)
- **`pod012`**: Solo el **10%** ha insultado/usado lenguaje obsceno (vs 20% global) → **-10pp** (sobre: 'NO' +10pp)
- **`pod010`**: **0%** ha hecho cosas malas a personas intencionalmente (vs 12% global) → **-12pp** (sobre: 'NO' 100% vs 88% global)

##### **Baja agresión física y peleas:**
- **`pad033`**: Solo el **7%** ha estado en una pelea (vs 21% global) → **-15pp** (sobre: 'NO' +15pp)
- **`pcd027`**: Solo el **12%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **-16pp** (sobre: 'NO' +16pp)
- **`pad025`**: **0%** ha trepado/corrido cuando no debía (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 88% global)

##### **Bajo vandalismo y daño a propiedad:**
- **`pcd022`**: Solo el **7%** ha iniciado un fuego sin permiso (vs 19% global) → **-11pp** (sobre: 'NO' +12pp)
- **`pcd020`**: **0%** ha destrozado un lugar intencionalmente (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 88% global)
- **`pcd021`**: **0%** ha dañado las cosas de otros a propósito (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 89% global)

##### **Bajo consumo de tabaco y ausencia de intentos suicidas:**
- **`pni001`**: Solo el **28%** ha fumado cigarrillos (vs 42% global) → **-14pp** (sobre: 'NO' +15pp)
- **`pmd022`**: **0%** ha intentado suicidarse (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 88% global)

#### Variables más distintivas (diferencias >10pp):

| Variable | Descripción | % Clase 1 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pod003** | Desobediencia intencional (NO) | 86% | 68% | **+18pp** |
| **pcd027** | Peleas físicas con lesiones (NO) | 88% | 72% | **+16pp** |
| **pad033** | Estar en una pelea (NO) | 93% | 78% | **+15pp** |
| **pod005** | Molestar a otros intencionalmente (NO) | 87% | 72% | **+15pp** |
| **pni001** | Fumar cigarrillos (NO) | 72% | 57% | **+15pp** |
| **pcd022** | Iniciar fuego sin permiso (NO) | 93% | 81% | **+12pp** |
| **pod010** | Hacer cosas malas a personas (NO) | 100% | 88% | **+12pp** |
| **pad025** | Trepar/correr cuando no debía (NO) | 100% | 88% | **+12pp** |
| **pmd022** | Intento de suicidio (NO) | 100% | 88% | **+12pp** |

#### Síntesis interpretativa:

La **Clase 1** representa el **grupo de referencia más saludable o de menor riesgo** dentro de la muestra de 6 clases, que se caracteriza por:

1. **Ausencia casi completa de problemas conductuales externalizantes**: muy baja desobediencia, agresión, vandalismo y conductas disruptivas.
2. **Ningún intento de suicidio** y baja prevalencia de conductas de riesgo asociadas.
3. **Consumo reducido de tabaco** en comparación con la muestra global.
4. **Múltiples variables con 0% de prevalencia** de problemas graves (daño a propiedad, incendios, agresión intencional).


### Clase 2 (k = 6)
  22 observaciones (3.0%)
  
  ---
  Variables distintivas:

     1. psz014:
        SOBRE 1: 'SI' - 55% (12/22) vs 3% (23/727) global, +51pp
        SUB 1: 'NO' - 41% (9/22) vs 96% (701/727) global, -56pp

     2. pmd022:
        SOBRE 1: 'SI' - 59% (13/22) vs 11% (83/727) global, +48pp
        SUB 1: 'NO' - 36% (8/22) vs 88% (641/727) global, -52pp

     3. pni001:
        SOBRE 1: 'SI' - 82% (18/22) vs 42% (307/727) global, +40pp
        SUB 1: 'NO' - 18% (4/22) vs 57% (417/727) global, -39pp

     4. pad033:
        SOBRE 1: 'SI' - 59% (13/22) vs 21% (156/727) global, +38pp
        SUB 1: 'NO' - 41% (9/22) vs 78% (568/727) global, -37pp

     5. pod012:
        SOBRE 1: 'SI' - 50% (11/22) vs 20% (145/727) global, +30pp
        SUB 1: 'NO' - 50% (11/22) vs 80% (579/727) global, -30pp

     6. pod005:
        SOBRE 1: 'SI' - 55% (12/22) vs 28% (202/727) global, +27pp
        SUB 1: 'NO' - 45% (10/22) vs 72% (522/727) global, -26pp

     7. pad025:
        SOBRE 1: 'SI' - 36% (8/22) vs 11% (83/727) global, +25pp
        SUB 1: 'NO' - 64% (14/22) vs 88% (641/727) global, -25pp

     8. pod011:
        SOBRE 1: 'SI' - 32% (7/22) vs 8% (57/727) global, +24pp
        SUB 1: 'NO' - 68% (15/22) vs 92% (667/727) global, -24pp

     9. pcd027:
        SOBRE 1: 'SI' - 50% (11/22) vs 27% (199/727) global, +23pp
        SUB 1: 'NO' - 50% (11/22) vs 72% (525/727) global, -22pp

    10. pcd029:
        SOBRE 1: 'SI' - 23% (5/22) vs 4% (26/727) global, +19pp
        SUB 1: 'NO' - 77% (17/22) vs 96% (699/727) global, -19pp

    11. pod003:
        SOBRE 1: 'SI' - 50% (11/22) vs 32% (230/727) global, +18pp
        SUB 1: 'NO' - 50% (11/22) vs 68% (494/727) global, -18pp

    12. pcd020:
        SOBRE 1: 'SI' - 27% (6/22) vs 11% (81/727) global, +16pp
        SUB 1: 'NO' - 73% (16/22) vs 88% (643/727) global, -16pp

    13. pcd022:
        SOBRE 1: 'SI' - 32% (7/22) vs 19% (136/727) global, +13pp
        SUB 1: 'NO' - 68% (15/22) vs 81% (588/727) global, -13pp

    14. pcd039:
        SOBRE 1: 'SI' - 18% (4/22) vs 7% (49/727) global, +11pp
        SUB 1: 'NO' - 82% (18/22) vs 93% (675/727) global, -11pp

#### Perfil general:
La **Clase 2** es un grupo muy pequeño pero extremadamente grave que representa solo el **3.0%** de la muestra (22/727 observaciones). Muestra un **perfil de psicosis grave con conducta suicida, agresividad y consumo de tabaco**, caracterizado por una prevalencia muy alta de síntomas psicóticos delirantes, intentos de suicidio, conductas disruptivas y agresivas, y tabaquismo extremo.

#### Hallazgos principales:

##### **Síntomas psicóticos graves (delirios de control):**
- **`psz014`**: **55%** está convencido de estar bajo el control de algún poder o fuerza (vs 3% global) → **+51pp**

##### **Conducta suicida grave:**
- **`pmd022`**: **59%** ha intentado suicidarse (vs 11% global) → **+48pp**

##### **Consumo extremo de tabaco:**
- **`pni001`**: **82%** ha fumado cigarrillos (vs 42% global) → **+40pp**

##### **Agresión física y peleas:**
- **`pad033`**: **59%** ha estado en una pelea (vs 21% global) → **+38pp**
- **`pcd027`**: **50%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **+23pp**
- **`pcd029`**: **23%** ha herido a alguien con un arma (vs 4% global) → **+19pp**

##### **Conducta disruptiva y oposicionista:**
- **`pod012`**: **50%** ha insultado/usado lenguaje obsceno (vs 20% global) → **+30pp**
- **`pod005`**: **55%** hace cosas para molestar/enojar a otros (vs 28% global) → **+27pp**
- **`pad025`**: **36%** ha trepado/corrido cuando no debía (vs 11% global) → **+25pp**
- **`pod011`**: **32%** se ha vengado de personas dañando sus cosas/hiriéndolas (vs 8% global) → **+24pp**
- **`pod003`**: **50%** ha hecho cosas que sus cuidadores le prohibieron a propósito (vs 32% global) → **+18pp**

##### **Vandalismo y problemas legales:**
- **`pcd020`**: **27%** ha destrozado un lugar intencionalmente (vs 11% global) → **+16pp**
- **`pcd022`**: **32%** ha iniciado un fuego sin permiso (vs 19% global) → **+13pp**
- **`pcd039`**: **18%** ha tenido problemas con la policía (vs 7% global) → **+11pp**

#### Variables más distintivas (diferencias >30pp):

| Variable | Descripción | % Clase 2 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **psz014** | Convencido de estar bajo control de un poder/fuerza  | 55% | 3% | **+51pp** |
| **pmd022** | Intento de suicidio  | 59% | 11% | **+48pp** |
| **pni001** | Fumar cigarrillos  | 82% | 42% | **+40pp** |
| **pad033** | Estar en una pelea  | 59% | 21% | **+38pp** |
| **pod012** | Insultar/lenguaje obsceno  | 50% | 20% | **+30pp** |
| **pod005** | Molestar a otros intencionalmente  | 55% | 28% | **+27pp** |

#### Síntesis interpretativa:

La **Clase 2** representa un **grupo de gravedad con prevalencia en psicosis, riesgo suicida y conducta violenta** que se caracteriza por:

1. **Síntomas psicóticos delirantes graves**: ideación de control externo, que sugiere un trastorno psicótico potencial.
2. **Alto riesgo suicida**: más de la mitad ha intentado suicidarse.
3. **Consumo casi universal de tabaco**: posiblemente como automedicación o como parte de un patrón de conductas de riesgo.
4. **Conducta agresiva y violenta**: alta participación en peleas, con una proporción significativa que ha usado armas.
5. **Conducta disruptiva y oposicionista marcada**: desobediencia, lenguaje obsceno, conductas vengativas.

### Clase 3 (k = 6)
  111 observaciones (15.3%)
  
  ---
  Variables distintivas:

     1. pmd022:
        SOBRE 1: 'SI' - 28% (31/111) vs 11% (83/727) global, +17pp
        SUB 1: 'NO' - 72% (80/111) vs 88% (641/727) global, -16pp

     2. pcd027:
        SOBRE 1: 'NO' - 85% (94/111) vs 72% (525/727) global, +12pp
        SUB 1: 'SI' - 15% (17/111) vs 27% (199/727) global, -12pp

     3. pad025:
        SOBRE 1: 'NO' - 100% (111/111) vs 88% (641/727) global, +12pp

     4. pcd020:
        SOBRE 1: 'NO' - 100% (111/111) vs 88% (643/727) global, +12pp

     5. pcd021:
        SOBRE 1: 'NO' - 100% (111/111) vs 89% (646/727) global, +11pp

     6. pod012:
        SOBRE 1: 'NO' - 90% (100/111) vs 80% (579/727) global, +10pp
        SUB 1: 'SI' - 10% (11/111) vs 20% (145/727) global, -10pp

#### Perfil general:
La **Clase 3** (k=6) es un grupo de tamaño mediano que representa el **15.3%** de la muestra (111/727 observaciones). Muestra un **perfil de riesgo suicida elevado con baja conducta externalizante**, caracterizado por una alta prevalencia de intentos de suicidio junto con una baja prevalencia de conductas agresivas, destructivas y disruptivas.

#### Hallazgos principales:

##### **Alta conducta suicida:**
- **`pmd022`**: **28%** ha intentado suicidarse (vs 11% global) → **+17pp**

##### **Baja agresión física:**
- **`pcd027`**: Solo el **15%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **-12pp** (sobre: 'NO' +12pp)

##### **Baja conducta disruptiva e hiperactiva:**
- **`pad025`**: **0%** ha trepado/corrido cuando no debía (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 88% global)
- **`pod012`**: Solo el **10%** ha insultado/usado lenguaje obsceno (vs 20% global) → **-10pp** (sobre: 'NO' +10pp)

##### **Ausencia de vandalismo y daño a propiedad:**
- **`pcd020`**: **0%** ha destrozado un lugar intencionalmente (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 88% global)
- **`pcd021`**: **0%** ha dañado las cosas de otros a propósito (vs 11% global) → **-11pp** (sobre: 'NO' 100% vs 89% global)

#### Variables más distintivas (diferencias >10pp):

| Variable | Descripción | % Clase 3 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pmd022** | Intento de suicidio (SI) | 28% | 11% | **+17pp** |
| **pad025** | Trepar/correr cuando no debía (NO) | 100% | 88% | **+12pp** |
| **pcd020** | Destruir propiedad intencionalmente (NO) | 100% | 88% | **+12pp** |
| **pcd027** | Peleas físicas con lesiones (NO) | 85% | 72% | **+12pp** |
| **pcd021** | Dañar cosas de otros a propósito (NO) | 100% | 89% | **+11pp** |
| **pod012** | Insultar/lenguaje obsceno (NO) | 90% | 80% | **+10pp** |

#### Síntesis interpretativa:

La **Clase 3** representa un **grupo con riesgo suicida significativo pero sin los problemas conductuales externalizantes típicos** que se caracteriza por:

1. **Alto riesgo suicida**: Casi un tercio ha intentado suicidarse, más del doble de la tasa global.
2. **Ausencia de conductas externalizantes graves**: No presenta vandalismo, daño a propiedad ni hiperactividad disruptiva.
3. **Baja agresión física**: Menos participación en peleas con lesiones que el promedio.
4. **Baja conducta disruptiva verbal**: Menos uso de lenguaje obsceno o insultos.

Esta clase representa un grupo particularmente vulnerable porque su sufrimiento no se manifiesta externamente a través de conductas problemáticas, lo que puede hacer que pasen desapercibidos en entornos escolares o comunitarios hasta que ocurre un intento de suicidio.  

### Clase 4 (k = 6)
  15 observaciones (2.1%)
  
  ---
  Variables distintivas:

     1. pmj001:
        SOBRE 1: 'SI' - 60% (9/15) vs 13% (98/727) global, +47pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 20% (3/15) vs 86% (626/727) global, -66pp

     2. pcd021:
        SOBRE 1: 'SI' - 53% (8/15) vs 11% (78/727) global, +43pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 27% (4/15) vs 89% (646/727) global, -62pp

     3. pcd020:
        SOBRE 1: 'SI' - 53% (8/15) vs 11% (81/727) global, +42pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 27% (4/15) vs 88% (643/727) global, -62pp

     4. pcd028:
        SOBRE 1: 'SI' - 47% (7/15) vs 7% (48/727) global, +40pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 33% (5/15) vs 93% (676/727) global, -60pp

     5. pcd029:
        SOBRE 1: 'SI' - 47% (7/15) vs 4% (26/727) global, +43pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 40% (6/15) vs 96% (699/727) global, -56pp

     6. pcd013:
        SOBRE 1: 'SI' - 40% (6/15) vs 6% (46/727) global, +34pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 40% (6/15) vs 93% (678/727) global, -53pp

     7. pcd039:
        SOBRE 1: 'SI' - 40% (6/15) vs 7% (49/727) global, +33pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 40% (6/15) vs 93% (675/727) global, -53pp

     8. pcd027:
        SOBRE 1: 'SI' - 80% (12/15) vs 27% (199/727) global, +53pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp

     9. pod005:
        SOBRE 1: 'SI' - 60% (9/15) vs 28% (202/727) global, +32pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 20% (3/15) vs 72% (522/727) global, -52pp

    10. pni001:
        SOBRE 1: 'SI' - 73% (11/15) vs 42% (307/727) global, +31pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 7% (1/15) vs 57% (417/727) global, -51pp

    11. pod003:
        SOBRE 1: 'SI' - 60% (9/15) vs 32% (230/727) global, +28pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 20% (3/15) vs 68% (494/727) global, -48pp

    12. pcd022:
        SOBRE 1: 'SI' - 47% (7/15) vs 19% (136/727) global, +28pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 33% (5/15) vs 81% (588/727) global, -48pp

    13. pcd008:
        SOBRE 1: 'SI' - 27% (4/15) vs 1% (4/727) global, +26pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 53% (8/15) vs 99% (720/727) global, -46pp

    14. psu011:
        SOBRE 1: 'SI' - 27% (4/15) vs 1% (4/727) global, +26pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 53% (8/15) vs 99% (720/727) global, -46pp

    15. pcd007:
        SOBRE 1: 'SI' - 20% (3/15) vs 0% (3/727) global, +20pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 60% (9/15) vs 99% (721/727) global, -39pp

    16. pcd009:
        SOBRE 1: 'SI' - 20% (3/15) vs 1% (4/727) global, +19pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 60% (9/15) vs 99% (720/727) global, -39pp

    17. pad033:
        SOBRE 1: 'SI' - 40% (6/15) vs 21% (156/727) global, +19pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 40% (6/15) vs 78% (568/727) global, -38pp

    18. pad025:
        SOBRE 1: 'SI' - 27% (4/15) vs 11% (83/727) global, +15pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 53% (8/15) vs 88% (641/727) global, -35pp

    19. pmd022:
        SOBRE 1: 'SI' - 33% (5/15) vs 11% (83/727) global, +22pp
        SUB 1: 'NO' - 53% (8/15) vs 88% (641/727) global, -35pp

    20. pod010:
        SOBRE 1: 'SI' - 27% (4/15) vs 12% (86/727) global, +15pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 53% (8/15) vs 88% (638/727) global, -34pp

    21. pod012:
        SOBRE 1: 'SI' - 33% (5/15) vs 20% (145/727) global, +13pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 47% (7/15) vs 80% (579/727) global, -33pp

    22. psu009:
        SOBRE 1: 'SI' - 13% (2/15) vs 0% (2/727) global, +13pp
        SOBRE 2: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SUB 1: 'NO' - 67% (10/15) vs 99% (722/727) global, -33pp

    23. pcd002:
        SOBRE 1: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SOBRE 2: 'SI' - 20% (3/15) vs 7% (51/727) global, +13pp
        SUB 1: 'NO' - 60% (9/15) vs 93% (673/727) global, -33pp

    24. pod011:
        SOBRE 1: 'nan' - 13% (2/15) vs 0% (2/727) global, +13pp
        SOBRE 2: 'SI' - 20% (3/15) vs 8% (57/727) global, +12pp
        SUB 1: 'NO' - 60% (9/15) vs 92% (667/727) global, -32pp

    25. psz014:
        SOBRE 1: 'SI' - 13% (2/15) vs 3% (23/727) global, +10pp
        SUB 1: 'NO' - 73% (11/15) vs 96% (701/727) global, -23pp

 #### Perfil general:
La **Clase 4**  es un grupo muy pequeño pero extremadamente grave que representa solo el **2.1%** de la muestra (15/727 observaciones). Muestra un **perfil de delincuencia grave con violencia extrema, consumo de drogas y conductas antisociales severas**, caracterizado por una prevalencia muy alta de delitos violentos, vandalismo, consumo de marihuana y tabaco, problemas legales y conducta disruptiva.

#### Hallazgos principales:

##### **Violencia extrema y uso de armas:**
- **`pcd027`**: **80%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **+53pp**
- **`pcd029`**: **47%** ha herido a alguien con un arma (vs 4% global) → **+43pp**
- **`pcd028`**: **47%** ha intentado lastimar gravemente a alguien (vs 7% global) → **+40pp**

##### **Vandalismo y daño a propiedad grave:**
- **`pcd021`**: **53%** ha dañado las cosas de otros a propósito (vs 11% global) → **+43pp**
- **`pcd020`**: **53%** ha destrozado un lugar intencionalmente (vs 11% global) → **+42pp**
- **`pcd022`**: **47%** ha iniciado un fuego sin permiso (vs 19% global) → **+28pp**

##### **Consumo de sustancias:**
- **`pmj001`**: **60%** ha usado marihuana (vs 13% global) → **+47pp**
- **`pni001`**: **73%** ha fumado cigarrillos (vs 42% global) → **+31pp**
- **`psu011`**: **27%** ha usado otras drogas (vs 1% global) → **+26pp**
- **`psu009`**: **13%** ha usado inhalantes (vs >1% global) → **+13pp**

##### **Robos y delitos contra la propiedad:**
- **`pcd008`**: **27%** ha asaltado/atacado para robar (vs 1% global) → **+26pp**
- **`pcd009`**: **20%** ha amenazado para robar (vs 1% global) → **+19pp**
- **`pcd007`**: **20%** ha robado bolsos o joyería (vs >1% global) → **+20pp**
- **`pcd002`**: **20%** ha hurtado en tiendas (vs 7% global) → **+13pp**

##### **Problemas legales y conductuales tempranos:**
- **`pcd039`**: **40%** ha tenido problemas con la policía (vs 7% global) → **+33pp**
- **`pcd013`**: **40%** tuvo problemas por llegar tarde a edad temprana (vs 6% global) → **+34pp**

##### **Conducta disruptiva y oposicionista:**
- **`pod005`**: **60%** hace cosas para molestar/enojar a otros (vs 28% global) → **+32pp**
- **`pod003`**: **60%** ha hecho cosas que sus cuidadores le prohibieron a propósito (vs 32% global) → **+28pp**
- **`pod012`**: **33%** ha insultado/usado lenguaje obsceno (vs 20% global) → **+13pp**
- **`pod010`**: **27%** ha hecho cosas malas a personas intencionalmente (vs 12% global) → **+15pp**
- **`pod011`**: **20%** se ha vengado de personas (vs 8% global) → **+12pp**

##### **Agresión física y peleas:**
- **`pad033`**: **40%** ha estado en una pelea (vs 21% global) → **+19pp**
- **`pad025`**: **27%** ha trepado/corrido cuando no debía (vs 11% global) → **+15pp**

##### **Intento de suicidio y síntomas psicóticos:**
- **`pmd022`**: **33%** ha intentado suicidarse (vs 11% global) → **+22pp**
- **`psz014`**: **13%** está convencido de estar bajo el control de algún poder o fuerza (vs 3% global) → **+10pp**

##### **Patrón de datos faltantes:**
- Múltiples variables muestran valores 'nan' sobretrepresentados (+13pp), lo que podría indicar falta de respuesta o no aplicabilidad en este grupo.

#### Variables más distintivas (diferencias >40pp):

| Variable | Descripción | % Clase 4 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pcd027** | Peleas físicas con lesiones  | 80% | 27% | **+53pp** |
| **pmj001** | Uso de marihuana  | 60% | 13% | **+47pp** |
| **pcd029** | Herir a alguien con un arma  | 47% | 4% | **+43pp** |
| **pcd021** | Dañar cosas de otros a propósito  | 53% | 11% | **+43pp** |
| **pcd020** | Destruir propiedad intencionalmente  | 53% | 11% | **+42pp** |
| **pcd028** | Intentar lastimar gravemente a alguien  | 47% | 7% | **+40pp** |
| **pcd013** | Problemas por llegar tarde a edad temprana  | 40% | 6% | **+34pp** |
| **pcd039** | Problemas con la policía  | 40% | 7% | **+33pp** |

#### Síntesis interpretativa:

La **Clase 4** representa un **grupo de delincuencia juvenil grave y violencia extrema con policonsumo de sustancias** que se caracteriza por:

1. **Patrón delictivo diversificado y grave**: robos con violencia, vandalismo, uso de armas y daño a propiedad.
2. **Consumo precoz y múltiple de sustancias**: marihuana, tabaco y otras drogas.
3. **Historial de problemas conductuales y legales tempranos**: problemas con la policía y transgresiones desde edad temprana.
4. **Alto riesgo de violencia hacia otros y hacia sí mismos**: intentos de suicidio y agresión física extrema.
5. **Posible trastorno de conducta severo** con características psicóticas leves.

Esta clase representa probablemente el **subgrupo de mayor riesgo de cronicidad en el sistema de justicia** y con necesidades complejas que requieren coordinación entre servicios de salud mental, justicia juvenil y servicios sociales. 

### Clase 5 (k = 6)
  163 observaciones (22.4%)
  
  ---
  Variables distintivas:

     1. pni001:
        SOBRE 1: 'SI' - 65% (106/163) vs 42% (307/727) global, +23pp
        SUB 1: 'NO' - 35% (57/163) vs 57% (417/727) global, -22pp

     2. pcd027:
        SOBRE 1: 'SI' - 50% (81/163) vs 27% (199/727) global, +22pp
        SUB 1: 'NO' - 50% (82/163) vs 72% (525/727) global, -22pp

     3. pcd020:
        SOBRE 1: 'SI' - 28% (45/163) vs 11% (81/727) global, +16pp
        SUB 1: 'NO' - 72% (118/163) vs 88% (643/727) global, -16pp

     4. pcd021:
        SOBRE 1: 'SI' - 26% (43/163) vs 11% (78/727) global, +16pp
        SUB 1: 'NO' - 74% (120/163) vs 89% (646/727) global, -15pp

     5. pod003:
        SOBRE 1: 'SI' - 47% (77/163) vs 32% (230/727) global, +16pp
        SUB 1: 'NO' - 53% (86/163) vs 68% (494/727) global, -15pp

     6. pmj001:
        SOBRE 1: 'SI' - 28% (46/163) vs 13% (98/727) global, +15pp
        SUB 1: 'NO' - 72% (117/163) vs 86% (626/727) global, -14pp

     7. pod005:
        SOBRE 1: 'SI' - 42% (69/163) vs 28% (202/727) global, +15pp
        SUB 1: 'NO' - 58% (94/163) vs 72% (522/727) global, -14pp

     8. pcd022:
        SOBRE 1: 'SI' - 33% (53/163) vs 19% (136/727) global, +14pp
        SUB 1: 'NO' - 67% (110/163) vs 81% (588/727) global, -13pp

     9. pad025:
        SOBRE 1: 'SI' - 25% (40/163) vs 11% (83/727) global, +13pp
        SUB 1: 'NO' - 75% (123/163) vs 88% (641/727) global, -13pp

    10. pcd002:
        SOBRE 1: 'SI' - 20% (32/163) vs 7% (51/727) global, +13pp
        SUB 1: 'NO' - 80% (131/163) vs 93% (673/727) global, -12pp

    11. pad033:
        SOBRE 1: 'SI' - 33% (54/163) vs 21% (156/727) global, +12pp
        SUB 1: 'NO' - 67% (109/163) vs 78% (568/727) global, -11pp

    12. pod010:
        SOBRE 1: 'SI' - 23% (38/163) vs 12% (86/727) global, +11pp
        SUB 1: 'NO' - 77% (125/163) vs 88% (638/727) global, -11pp

    13. pod012:
        SOBRE 1: 'SI' - 31% (50/163) vs 20% (145/727) global, +11pp
        SUB 1: 'NO' - 69% (113/163) vs 80% (579/727) global, -10pp

    14. pcd028:
        SOBRE 1: 'SI' - 17% (28/163) vs 7% (48/727) global, +11pp
        SUB 1: 'NO' - 83% (135/163) vs 93% (676/727) global, -10pp

    15. pcd039:
        SOBRE 1: 'SI' - 17% (28/163) vs 7% (49/727) global, +10pp
        SUB 1: 'NO' - 83% (135/163) vs 93% (675/727) global, -10pp

    16. pcd013:
        SOBRE 1: 'SI' - 17% (27/163) vs 6% (46/727) global, +10pp

#### Perfil general:
La **Clase 5**  es el segundo grupo más grande con el **22.4%** de la muestra (163/727 observaciones). Muestra un **perfil de externalización moderada con consumo de sustancias y conducta disruptiva**, caracterizado por una prevalencia elevada de tabaquismo, agresión física, vandalismo, desobediencia y consumo de marihuana.

#### Hallazgos principales:

##### **Consumo elevado de tabaco y marihuana:**
- **`pni001`**: **65%** ha fumado cigarrillos (vs 42% global) → **+23pp**
- **`pmj001`**: **28%** ha usado marihuana (vs 13% global) → **+15pp**

##### **Agresión física frecuente:**
- **`pcd027`**: **50%** ha estado en una pelea física donde alguien salió herido (vs 27% global) → **+22pp**
- **`pad033`**: **33%** ha estado en una pelea (vs 21% global) → **+12pp**
- **`pcd028`**: **17%** ha intentado lastimar gravemente a alguien (vs 7% global) → **+11pp**

##### **Vandalismo y daño a propiedad:**
- **`pcd020`**: **28%** ha destrozado un lugar intencionalmente (vs 11% global) → **+16pp**
- **`pcd021`**: **26%** ha dañado las cosas de otros a propósito (vs 11% global) → **+16pp**
- **`pcd022`**: **33%** ha iniciado un fuego sin permiso (vs 19% global) → **+14pp**
- **`pcd002`**: **20%** ha hurtado en tiendas (vs 7% global) → **+13pp**

##### **Conducta disruptiva y oposicionista:**
- **`pod003`**: **47%** ha hecho cosas que sus cuidadores le prohibieron a propósito (vs 32% global) → **+16pp**
- **`pod005`**: **42%** hace cosas para molestar/enojar a otros (vs 28% global) → **+15pp**
- **`pad025`**: **25%** ha trepado/corrido cuando no debía (vs 11% global) → **+13pp**
- **`pod012`**: **31%** ha insultado/usado lenguaje obsceno (vs 20% global) → **+11pp**
- **`pod010`**: **23%** ha hecho cosas malas a personas intencionalmente (vs 12% global) → **+11pp**

##### **Problemas legales y transgresiones:**
- **`pcd039`**: **17%** ha tenido problemas con la policía (vs 7% global) → **+10pp**
- **`pcd013`**: **17%** tuvo problemas por llegar tarde a edad temprana (vs 6% global) → **+10pp**

#### Variables más distintivas (diferencias >15pp):

| Variable | Descripción | % Clase 5 | % Global | Diferencia |
|----------|-------------|-----------|----------|------------|
| **pni001** | Fumar cigarrillos  | 65% | 42% | **+23pp** |
| **pcd027** | Peleas físicas con lesiones  | 50% | 27% | **+22pp** |
| **pcd020** | Destruir propiedad intencionalmente  | 28% | 11% | **+16pp** |
| **pcd021** | Dañar cosas de otros a propósito  | 26% | 11% | **+16pp** |
| **pod003** | Desobediencia intencional  | 47% | 32% | **+16pp** |
| **pmj001** | Uso de marihuana  | 28% | 13% | **+15pp** |
| **pod005** | Molestar a otros intencionalmente  | 42% | 28% | **+15pp** |

#### Síntesis interpretativa:

La **Clase 5** representa un **grupo de externalización moderada con consumo de sustancias y problemas conductuales** que se caracteriza por:

1. **Consumo significativo de tabaco y marihuana**: uso de sustancias más elevado que el promedio.
2. **Agresión física frecuente**: alta participación en peleas con lesiones.
3. **Vandalismo y daño a propiedad**: conductas destructivas hacia la propiedad ajena.
4. **Conducta disruptiva y oposicionista**: desobediencia, molestia intencional a otros y lenguaje obsceno.
5. **Problemas legales moderados**: algunos problemas con la policía y transgresiones tempranas.

**Nota**: Esta clase muestra un **perfil externalizante de moderada gravedad** que no alcanza la extremidad de la Clase 4 pero supera significativamente los niveles de la Clase 1 (bajo riesgo). La combinación de consumo de sustancias con conductas agresivas y destructivas sugiere un patrón de riesgo que podría escalar sin intervención.

Esta clase representa un **grupo de riesgo intermedio** donde las intervenciones tempranas podrían ser particularmente efectivas para prevenir la progresión a perfiles más graves como el de la Clase 4.

## Síntesis de Diferenciación de Clases (k=6)

| **Criterio** | **Clase 0 (Externalización Grave Múltiple)** | **Clase 1 (Bajo Riesgo / Saludable)** | **Clase 2 (Psicosis Grave con Riesgo Suicida)** | **Clase 3 (Riesgo Suicida Internalizante Puro)** | **Clase 4 (Delincuencia Grave con Violencia Extrema)** | **Clase 5 (Externalización Moderada con Consumo)** |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Tamaño de la Clase** | 78 observaciones (10.7%) | 338 observaciones (46.5%) | 22 observaciones (3.0%) | 111 observaciones (15.3%) | 15 observaciones (2.1%) | 163 observaciones (22.4%) |
| **Patrón Principal** | **Externalización grave con múltiples problemas conductuales y agresivos.** Combinación de conducta disruptiva, agresión física, vandalismo, conducta vengativa, intentos de suicidio y tabaquismo. | **Ajuste adaptativo y baja psicopatología.** Ausencia casi completa de problemas externalizantes, conductas de riesgo y síntomas internalizantes graves. Actúa como grupo de referencia normativo. | **Psicosis grave con conducta suicida, agresividad y tabaquismo extremo.** Síntomas psicóticos delirantes, alto riesgo suicida, conducta violenta y consumo muy elevado de tabaco. | **Riesgo suicida elevado sin comorbilidad externalizante.** Perfil internalizante puro con intentos de suicidio, pero sin agresión, vandalismo o conducta disruptiva significativa. | **Delincuencia juvenil grave con violencia extrema y policonsumo de sustancias.** Patrón delictivo diversificado, consumo precoz de drogas, problemas legales y conducta antisocial severa. | **Externalización moderada con consumo de sustancias y problemas conductuales.** Combinación de tabaquismo, agresión física, vandalismo, desobediencia y consumo de marihuana en niveles intermedios. |
| **Núcleo del Perfil** | **Conducta oposicionista disruptiva con agresión y riesgo dual.** Predominan la desobediencia intencional, las conductas para molestar a otros, las peleas físicas, el daño a propiedad y las conductas vengativas, junto con intentos de suicidio y tabaquismo. | **Resiliencia y factores protectores.** Se caracteriza por la **sub-representación** marcada de síntomas problemáticos, con prevalencia cercana a cero en conductas graves y riesgo suicida. Bajo consumo de tabaco. | **Trastorno psicótico con riesgo vital.** Núcleo de delirios de control (sentirse bajo un poder externo), intentos de suicidio muy frecuentes, agresividad y un patrón de tabaquismo casi universal. | **Sufrimiento internalizante silencioso.** Alto riesgo de suicidio sin manifestaciones externalizantes que alerten del problema. Ausencia de señales conductuales evidentes de malestar. | **Trastorno de conducta de inicio temprano y severidad extrema.** Combinación de violencia interpersonal (uso de armas, lesiones graves), vandalismo, robos con violencia, consumo de marihuana y otras drogas, e implicación temprana con la justicia. | **Patrón de riesgo externalizante escalable.** Consumo significativo de tabaco y marihuana, agresión física frecuente, vandalismo y desobediencia, que sin intervención podría progresar a un perfil más grave. |
| **Síntomas más Distintivos (Diferencia >15pp vs. Global)** | 1. **pad033:** Peleas (+34pp)<br>2. **pad025:** Hiperactividad disruptiva (+28pp)<br>3. **pod005:** Molestar a otros (+26pp)<br>4. **pod003:** Desobediencia (+25pp)<br>5. **pod012:** Lenguaje obsceno (+24pp)<br>6. **pcd027:** Peleas con lesiones (+21pp)<br>7. **pod011:** Conducta vengativa (+20pp)<br>8. **pcd022:** Iniciar fuegos (+20pp) | 1. **pod003:** Baja desobediencia (-17pp en SI)<br>2. **pcd027:** Bajas peleas con lesiones (-16pp)<br>3. **pad033:** Bajas peleas (-15pp)<br>4. **pod005:** Bajo molestar a otros (-14pp)<br>5. **pni001:** Bajo tabaquismo (-14pp)<br>6. **pmd022:** Sin intentos de suicidio (-11pp) | 1. **psz014:** Delirios de control (+51pp)<br>2. **pmd022:** Intentos de suicidio (+48pp)<br>3. **pni001:** Tabaquismo extremo (+40pp)<br>4. **pad033:** Peleas (+38pp)<br>5. **pod012:** Lenguaje obsceno (+30pp)<br>6. **pod005:** Molestar a otros (+27pp) | 1. **pmd022:** Intentos de suicidio (+17pp)<br>2. **pcd027:** Bajas peleas con lesiones (-12pp)<br>3. **pad025:** Sin hiperactividad disruptiva (-11pp)<br>4. **pcd020:** Sin destruir propiedad (-11pp)<br>5. **pcd021:** Sin dañar cosas de otros (-11pp) | 1. **pcd027:** Peleas con lesiones (+53pp)<br>2. **pmj001:** Uso de marihuana (+47pp)<br>3. **pcd029:** Herir con arma (+43pp)<br>4. **pcd021:** Dañar cosas de otros (+43pp)<br>5. **pcd020:** Destruir propiedad (+42pp)<br>6. **pcd028:** Intentar lastimar gravemente (+40pp)<br>7. **pcd013:** Problemas tempranos por llegar tarde (+34pp)<br>8. **pcd039:** Problemas con la policía (+33pp) | 1. **pni001:** Tabaquismo (+23pp)<br>2. **pcd027:** Peleas con lesiones (+22pp)<br>3. **pcd020:** Destruir propiedad (+16pp)<br>4. **pcd021:** Dañar cosas de otros (+16pp)<br>5. **pod003:** Desobediencia (+16pp)<br>6. **pmj001:** Uso de marihuana (+15pp)<br>7. **pod005:** Molestar a otros (+15pp) |
| **Riesgo Asociado** | **Alto.** Riesgo de daño a otros (agresión, vandalismo), autolesión (suicidio), problemas legales y escolares, y desarrollo de dependencia al tabaco. | **Muy Bajo.** Representa el perfil de mayor ajuste y salud dentro de la muestra. El riesgo es la posible transición a clases problemáticas sin factores protectores mantenidos. | **Muy Alto.** Riesgo vital inmediato por suicidio, posible comportamiento violento impulsivo vinculado a delirios, y deterioro funcional grave por psicosis no tratada. | **Alto.** Riesgo de suicidio elevado y particularmente peligroso por la falta de señales externalizantes de alerta, lo que dificulta la identificación y intervención temprana. | **Extremo.** Alto riesgo de cronicidad en el sistema de justicia, victimización o perpetración de violencia grave, complicaciones por abuso de sustancias, y conducta suicida en un contexto de alta impulsividad. | **Moderado-Alto.** Riesgo de escalada en el consumo de sustancias (a drogas más duras), de que los problemas conductuales se agraven hacia la delincuencia, y de implicación con el sistema de justicia. |
| **Abordaje Sugerido** | **Intervención conductual intensiva y manejo de riesgo dual.** Programas para control de impulsos, manejo de la ira, prevención del vandalismo, tratamiento del riesgo suicida y cesación tabáquica. Intervenciones familiares para supervisión y establecimiento de límites. | **Prevención universal y promoción de la salud.** Educación psico-social para mantener factores protectores, desarrollo de habilidades para la vida y monitoreo para la detección precoz de cualquier problema emergente. | **Hospitalización y tratamiento psiquiátrico urgente.** Estabilización del riesgo suicida y homicida, tratamiento con antipsicóticos, terapia para síntomas psicóticos, manejo de la agresividad y tratamiento para la dependencia del tabaco. | **Intervención centrada en el riesgo suicida y la depresión.** Evaluación exhaustiva y seguimiento estrecho, psicoterapia para la depresión y desesperanza, entrenamiento en regulación emocional, y psicoeducación familiar para reconocer señales de alarma sutiles. | **Tratamiento multidisciplinario intensivo en contexto de justicia juvenil.** Intervenciones judiciales especializadas, tratamiento residencial para conductas antisociales, desintoxicación y rehabilitación por uso de sustancias, y manejo de la agresividad y el riesgo suicida. | **Intervención temprana y preventiva para reducir la escalada.** Programas de prevención y tratamiento del consumo de tabaco y marihuana, terapia conductual para reducir la desobediencia y agresión, actividades prosociales alternativas y fortalecimiento de habilidades parentales. |
 
  ---

In [20]:
guardar_df(final_df_child, 'LCA_child', 'LCA')
guardar_df(final_df_teen, 'LCA_teen', 'LCA')
guardar_df(prob_df_child, 'LCA_prob_child', 'LCA_prob')
guardar_df(prob_df_teen, 'LCA_prob_teen', 'LCA_prob')

✓ DataFrame guardado como:
  • CSV: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA\csv\LCA_child.csv
  • Excel: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA\excel\LCA_child.xlsx
✓ DataFrame guardado como:
  • CSV: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA\csv\LCA_teen.csv
  • Excel: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA\excel\LCA_teen.xlsx
✓ DataFrame guardado como:
  • CSV: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA_prob\csv\LCA_prob_child.csv
  • Excel: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA_prob\excel\LCA_prob_child.xlsx
✓ DataFrame guardado como:
  • CSV: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\resultados\LCA_prob\csv\LCA_prob_teen.csv
  • Excel: C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_Tes